In [1]:
from __future__ import division
import pandas as pd
import numpy as np
from copy import deepcopy

import warnings
warnings.filterwarnings('ignore')

import py_stringmatching as sm
import py_entitymatching as em
import py_stringsimjoin as ssj

from py_entitymatching.catalog import catalog_manager as cm

import random

# Introduzione

Questo Notebook riassume tutto il processo di Entity Resolution:
Blocking, Matching e Clustering.


Contiene i punti essenziali per discutere  l' Entity Resolution



# Funzioni utilizzate

In [2]:
######################CLUSTERING######################

import networkx as nx

def ClusterComponentiConnessi(MatchTable, TuttiInodi):

    MatchTable=deepcopy(MatchTable)
    MatchTable.columns=['A','B']

    Singleton = set(TuttiInodi) - set(MatchTable['A']).union(set(MatchTable['B']))

    # Creazione del grafo a partire dagli elementi della MatchTable
    G = nx.Graph()
    for _, row in MatchTable.iterrows():
        G.add_edge(row['A'], row['B'])
#        G.add_edge(row['A'], row['B'], weight=row['sim'])  # Aggiungi il peso (etichetta) basato su 'sim'

    # Aggiungi gli elementi singleton all'insieme dei nodi
    for element in Singleton:
        G.add_node(element)

    # Calcola i componenti connessi (clusters)
    clusters = list(nx.connected_components(G))

    # Creazione del DataFrame dei cluster
    cluster_data = {'ClusterKey': [], 'ClusterElement': []}
    for i, cluster in enumerate(clusters):
        for element in cluster:
            cluster_data['ClusterKey'].append(i + 1)
            cluster_data['ClusterElement'].append(element)

    cluster_df = pd.DataFrame(cluster_data)
    return cluster_df

In [3]:
def VisualizzaCluster(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS

def VisualizzaDistribuzioneCluster(Clusters):

    val = VisualizzaCluster(Clusters)
    Risultato=val.groupby('#Record')['ClusterKey'].nunique().reset_index()
    Risultato.columns=['#Record','#Cluster']
    Risultato=Risultato[['#Cluster','#Record']]
    return Risultato.sort_values('#Record', ascending=False)

def _VisualizzaDistribuzioneCluster(Clusters):
    gruppi = Clusters.groupby('ClusterKey')
    conteggio_gruppi = gruppi.size().reset_index(name='NumeroElementiPerCluster')
    Risultato=conteggio_gruppi.groupby('NumeroElementiPerCluster').size().reset_index(name='NumeroCluster')
    print("Numero Elementi", (Risultato['NumeroElementiPerCluster'] * Risultato['NumeroCluster']).sum())
    ClusterMax=conteggio_gruppi[conteggio_gruppi['NumeroElementiPerCluster']==Risultato['NumeroElementiPerCluster'].max()]
    print("Cluster con max numero di elementi:", ClusterMax['ClusterKey'].tolist())

    return Risultato

In [4]:
def CalcolaMatchIndottiCluster(Cluster):
  Join=pd.merge(Cluster,Cluster, on='ClusterKey')
  Join=Join[Join.ClusterElement_x<Join.ClusterElement_y]
  Join=Join[['ClusterElement_x','ClusterElement_y']]
  Join.columns=['l_id','r_id']

  return Join.drop_duplicates()

In [5]:
def stable_marriage(MatchTable:pd.DataFrame):
    MATCH = pd.DataFrame(columns=['l_id', 'r_id', "sim"])
    MT = deepcopy(MatchTable)
    MT = MT.sort_values(["sim"], ascending=[False])
    while True:
        R = MT.loc[(~MT['l_id'].isin(MATCH['l_id'])) & (~MT['r_id'].isin(MATCH['r_id']))]
        if len(R) == 0:
            break
        x = R.iloc[0,:]
        MATCH = MATCH.append(x, ignore_index=True)
    return MATCH

def simmetric_best_match(MatchTable:pd.DataFrame):
  CMT = deepcopy(MatchTable)

  CMT['A_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['l_id']) \
             .cumcount() + 1

  CMT['B_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['r_id']) \
             .cumcount() + 1

  return CMT[(CMT.A_RowNo==1) & (CMT.B_RowNo==1)].drop(columns=['A_RowNo', 'B_RowNo']).sort_values(['sim'], ascending=[False])

In [6]:
def Valuta2(Gold:pd.DataFrame, Match:pd.DataFrame):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta2(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [7]:
def Valuta(Gold:pd.DataFrame, Match:pd.DataFrame):
 #   Gold = Gold[['l_id','r_id']]
 #   Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    #Gold = Gold[['l_id','r_id']]
    #Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [8]:
def ValutaBlocking(DA,DB,Block,Gold):
# INPUT : entrambi Block (è il candidate set after blocking)
#        e Gold (Gold Standard) devono essere con due colonne, l_id e r_id
# per avere indipendenza dal nome di queste colonne
# Si suppone che in Block l_id e r_id siano rispettivamente la seconda e la terza colonna
# e che in Gold sia la prima e la seconda
  Block = Block.iloc[:, [1, 2]].copy()
  Gold = Gold.iloc[:, [0, 1]].copy()
  Gold.columns=Block.columns=['l_id','r_id']

  JOIN=pd.merge(Gold, Block)

 # Reduction_Ratio
  RR=1-len(Block)/(len(DA)*len(DB))
 # Pairs Completeness o Recall
  PC = len(JOIN)/len(Gold)
 # Pairs Quality
  PQ = len(JOIN)/len(Block)

  Risultato = pd.DataFrame([(DA.shape[0],DB.shape[0],Block.shape[0],round(RR,4),round(PC,4),round(PQ,4))],
                             columns=['A', 'B', 'BlockSize', 'ReductRatio','PCompletness','PQuality'])

  return Risultato

In [9]:
def IdSOURCES(Sources:list):
  ListaID= []
  for s in Sources.keys():
    ListaID += Sources[s]['id'].to_list()
  return ListaID

In [10]:
A=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/fodors.csv')
B=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/zagats.csv')
GoldStandard=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/matches_fodors_zagats.csv')
GoldStandard.columns=['l_id','r_id']
A=A.rename(columns={'id': 'l_id'})
B=B.rename(columns={'id': 'r_id'})

GoldStandard['id']=range(len(GoldStandard))
em.set_key(GoldStandard, 'id')

em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

cm.set_candset_properties(GoldStandard, 'id', 'l_id', 'r_id', A, B)

True

## Confronto RuleBasedBlocker e BooleanRuleMatcher


Per confrontare logica del RuleBasedBlocker
con quella del BooleanRuleMatcher,
consideriamo l'esempio già fatto del RuleBasedBlocker



```
rb = em.RuleBasedBlocker()
rb.add_rule(['name_name_jac_qgm_3_qgm_3(ltuple, rtuple) < 0.3',
             'phone_phone_lev_sim(ltuple, rtuple) < 0.8'], block_features)
rb.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) < 0.1'], block_features)
```


e otteniamo le stesse coppie con BooleanRuleMatcher.


### RuleBasedBlocker

In [11]:
#   BLOCCO LE COPPIE CON name_name_jac_qgm_3_qgm_3 < 0.3   AND phone_phone_lev_sim < 0.8
#rb.add_rule(['name_name_jac_qgm_3_qgm_3(ltuple, rtuple) < 0.3','phone_phone_lev_sim(ltuple, rtuple) < 0.8'], block_features)
#   QUINDI PASSANO QUELLE CON phone_phone_lev_sim >= 0.8. OR name_name_jac_qgm_3_qgm_3 >= 0.3

# AGGIUNGO  la seguente regola
# rb.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) < 0.1'], block_features)
#   BLOCCO LE COPPIE CON
#   PASSANO QUELLE CON addr_addr_jac_qgm_3_qgm_3 >= 0.1

# add_rule effettua l'OR: BLOCCO LE COPPIE CON
#    name_name_jac_qgm_3_qgm_3 < 0.3   AND phone_phone_lev_sim < 0.8
#    OPPURE
#    addr_addr_jac_qgm_3_qgm_3 < 0.1

#   QUINDI PASSANO LE COPPIE CON
#   (phone_phone_lev_sim >= 0.8. OR name_name_jac_qgm_3_qgm_3 >= 0.3)
#   AND  (addr_addr_jac_qgm_3_qgm_3 >= 0.1)

In [12]:
CorrispondenzeAttributi = em.get_attr_corres(A, B)
atypesA = em.get_attr_types(A)
atypesB = em.get_attr_types(B)

# per uniformare metto il tipo in B di phone uquale a quello in A
atypesB['phone']=atypesA['name']

block_features = em.get_features(A, B,
                                 atypesA, atypesB,
                                 CorrispondenzeAttributi,
                                 em.get_tokenizers_for_blocking(),
                                 em.get_sim_funs_for_blocking())

In [13]:
rb = em.RuleBasedBlocker()
rb.add_rule(['name_name_jac_qgm_3_qgm_3(ltuple, rtuple) < 0.3',
             'phone_phone_lev_sim(ltuple, rtuple) < 0.8'], block_features)
rb.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) < 0.1'], block_features)
C = rb.block_tables(A, B, l_output_attrs=['name'], r_output_attrs=['name'], show_progress=False)


In [14]:
ValutaBlocking(A,B,C,GoldStandard)


,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,532,330,152,0.9991,0.973,0.7105


In [15]:
# considero come match il risultato del blocking (LE COPPIE che passano)
#   (phone_phone_lev_sim >= 0.8. OR name_name_jac_qgm_3_qgm_3 >= 0.3)  AND  (addr_addr_jac_qgm_3_qgm_3 >= 0.1)
Valuta(GoldStandard,C)

,MT,TP,FP,FN,P,R,F
0,152,108,44,3,0.7105,0.973,0.8213


### BooleanRuleMatcher

Come ottenere (senza blocking, quindi partendo dal prodotto cartesiano)

(phone_phone_lev_sim >= 0.8. OR name_name_jac_qgm_3_qgm_3 >= 0.3)  AND  (addr_addr_jac_qgm_3_qgm_3 >= 0.1)

con un BooleanRuleMatcher?

Si riscrive in forma equivalente
(phone_phone_lev_sim >= 0.8 AND  (addr_addr_jac_qgm_3_qgm_3 >= 0.1)
OR
(name_name_jac_qgm_3_qgm_3 >= 0.3)  AND  (addr_addr_jac_qgm_3_qgm_3 >= 0.1)



In [16]:
PCC = A.assign(key=1).merge(B.assign(key=1), on='key').drop('key', 1)

PCC['id']=range(len(PCC))
em.set_key(PCC, 'id')

em.set_ltable(PCC, A)
em.set_rtable(PCC, B)
em.set_fk_ltable(PCC, 'l_id')
em.set_fk_rtable(PCC, 'r_id')



True

In [17]:
CorrispondenzeAttributi = em.get_attr_corres(A, B)
atypesA = em.get_attr_types(A)
atypesB = em.get_attr_types(B)

# per uniformare metto il tipo in B di phone uquale a quello in A
atypesB['phone']=atypesA['name']

feature_table = em.get_features(A, B,
                                 atypesA, atypesB,
                                 CorrispondenzeAttributi,
                                 em.get_tokenizers_for_matching(),
                                 em.get_sim_funs_for_matching())

In [18]:
## attenzione , richiede alcuni minuti ...
brm = em.BooleanRuleMatcher()

brm.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.1',
             'phone_phone_lev_sim(ltuple, rtuple) >= 0.8'], feature_table)
brm.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.1',
              'name_name_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.3', ], feature_table)

predictions = brm.predict(PCC, target_attr='pred_label',  append=True)
MT=predictions[predictions.pred_label==1]

In [19]:
MT=predictions[predictions.pred_label==1]
# Porta 'id' come prima colonna
cols = ['id'] + [col for col in MT.columns if col != 'id']
MT = MT[cols]


In [20]:
MT=MT[['id', 'l_id', 'r_id', 'name_x', 'addr_x', 'city_x', 'phone_x', 'type_x', 
       'name_y', 'addr_y', 'city_y', 'phone_y', 'type_y', 'pred_label']]

In [21]:
Valuta(GoldStandard.astype(str),MT.astype(str))

,MT,TP,FP,FN,P,R,F
0,152,108,44,3,0.7105,0.973,0.8213


### Conclusioni

Si può ottenere lo stesso risultato, considerando la differente logica


Il RuleBasedBlocker è implementato tramite similarity join, quindi molto più efficiente. Verifichiamolo nel seguente semplice esempio:


In [22]:
# tramite RuleBasedBlocker
rb = em.RuleBasedBlocker()
rb.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) < 0.1'], block_features)
C = rb.block_tables(A, B, l_output_attrs=['name'], r_output_attrs=['name'], show_progress=False)
print(ValutaBlocking(A,B,C,GoldStandard))
print(Valuta(GoldStandard,C))

     A    B  BlockSize  ReductRatio  PCompletness  PQuality
0  532  330      41714       0.7624         0.973    0.0026
      MT   TP     FP  FN       P      R       F
0  41714  108  41606   3  0.0026  0.973  0.0052


In [23]:
# tramite Similarity Join
C_SIMJOIN_JAC  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                  'addr', 'addr',  sm.QgramTokenizer(qval=3), threshold=0.1,
                                  l_out_attrs=['name'],
                                  r_out_attrs=['name'])

ValutaBlocking(A,B,C_SIMJOIN_JAC,GoldStandard)


,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,532,330,41714,0.7624,0.973,0.0026


# Esempio

In [24]:
A=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/fodors.csv')
B=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/zagats.csv')
GoldStandard=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/matches_fodors_zagats.csv')
GoldStandard.columns=['l_id','r_id']
A=A.rename(columns={'id': 'l_id'})
B=B.rename(columns={'id': 'r_id'})

GoldStandard['id']=range(len(GoldStandard))
em.set_key(GoldStandard, 'id')

em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

cm.set_candset_properties(GoldStandard, 'id', 'l_id', 'r_id', A, B)

True

In [25]:
A['mix']=' '
B['mix']=' '
for x in ['name', 'addr' , 'city']:
    A['mix'] += A[x] + ' '
    B['mix'] += B[x] + ' '

C  = ssj.jaccard_join(A, B, 
                      'l_id', 'r_id', 
                      'mix', 'mix',  
                      sm.QgramTokenizer(qval=3), threshold=0.3,
                                  l_out_attrs=['name', 'addr' , 'city'],
                                  r_out_attrs=['name', 'addr' , 'city'])

C

,_id,l_l_id,r_r_id,l_name,l_addr,l_city,r_name,r_addr,r_city,_sim_score
0,0,654,5,broadway deli,3rd st. promenade,santa monica,benitas frites,1433 third st. promenade,santa monica,0.444444
1,1,696,5,remi,3rd st. promenade,santa monica,benitas frites,1433 third st. promenade,santa monica,0.484375
2,2,547,6,locanda veneta,3rd st.,los angeles,bernards,515 s. olive st.,los angeles,0.300000
3,3,553,6,rex il ristorante,617 s. olive st.,los angeles,bernards,515 s. olive st.,los angeles,0.421875
4,4,653,6,border grill,4th st.,los angeles,bernards,515 s. olive st.,los angeles,0.327586
...,...,...,...,...,...,...,...,...,...,...
2827,2827,623,331,cafe ritz-carlton buckhead,3434 peachtree rd.,atlanta,ritz-carlton cafe (atlanta),181 peachtree st.,atlanta,0.480000
2828,2828,624,331,dining room ritz-carlton buckhead,3434 peachtree rd.,atlanta,ritz-carlton cafe (atlanta),181 peachtree st.,atlanta,0.372093
2829,2829,625,331,restaurant ritz-carlton atlanta,181 peachtree st.,atlanta,ritz-carlton cafe (atlanta),181 peachtree st.,atlanta,0.621212
2830,2830,920,331,bistango,1100 peachtree st.,atlanta,ritz-carlton cafe (atlanta),181 peachtree st.,atlanta,0.358209


In [26]:
ValutaBlocking(A,B,C,GoldStandard)


,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,532,330,2832,0.9839,0.991,0.0388


In [27]:
C.sample(5)

,_id,l_l_id,r_r_id,l_name,l_addr,l_city,r_name,r_addr,r_city,_sim_score
1845,1845,731,265,bruno,240 e. 58th st.,new york,march,405 e. 58th st.,new york city,0.431373
2237,2237,629,315,aqua,252 california st.,san francisco,boulevard,1 mission st.,san francisco,0.322581
2732,2732,1007,327,grand cafe hotel monaco,501 geary st.,san francisco,plumpjack cafe,3127 fillmore st.,san francisco,0.303797
712,712,635,195,fringale,570 4th st.,san francisco,ebisu,1283 ninth ave.,san francisco,0.310345
2233,2233,1058,314,vivande porta via,2125 fillmore st.,san francisco,aqua,252 california st.,san francisco,0.309859


In [28]:
# Codice completo e generale per ottenere le  feature_table
block_c = em.get_attr_corres(A, B)
atypesA = em.get_attr_types(A)
atypesB = em.get_attr_types(B)
for c in A.columns:
    atypesB[c]=atypesA[c]

feature_table = em.get_features(A, B,
                                 atypesA, atypesB,
                                 block_c,
                                 em.get_tokenizers_for_matching(),
                                 em.get_sim_funs_for_matching())
print(feature_table['feature_name'].to_list())

['name_name_jac_qgm_3_qgm_3', 'name_name_cos_dlm_dc0_dlm_dc0', 'name_name_jac_dlm_dc0_dlm_dc0', 'name_name_mel', 'name_name_lev_dist', 'name_name_lev_sim', 'name_name_nmw', 'name_name_sw', 'addr_addr_jac_qgm_3_qgm_3', 'addr_addr_cos_dlm_dc0_dlm_dc0', 'addr_addr_jac_dlm_dc0_dlm_dc0', 'addr_addr_mel', 'addr_addr_lev_dist', 'addr_addr_lev_sim', 'addr_addr_nmw', 'addr_addr_sw', 'city_city_jac_qgm_3_qgm_3', 'city_city_cos_dlm_dc0_dlm_dc0', 'city_city_jac_dlm_dc0_dlm_dc0', 'city_city_mel', 'city_city_lev_dist', 'city_city_lev_sim', 'city_city_nmw', 'city_city_sw', 'phone_phone_jac_qgm_3_qgm_3', 'phone_phone_cos_dlm_dc0_dlm_dc0', 'phone_phone_jac_dlm_dc0_dlm_dc0', 'phone_phone_mel', 'phone_phone_lev_dist', 'phone_phone_lev_sim', 'phone_phone_nmw', 'phone_phone_sw', 'type_type_jac_qgm_3_qgm_3', 'type_type_cos_dlm_dc0_dlm_dc0', 'type_type_jac_dlm_dc0_dlm_dc0', 'type_type_mel', 'type_type_lev_dist', 'type_type_lev_sim', 'type_type_nmw', 'type_type_sw', 'mix_mix_jac_qgm_3_qgm_3', 'mix_mix_cos_dlm

In [29]:
C = C.rename(columns={
    'l_l_id': 'l_id',
    'r_r_id': 'r_id',
    '_id': 'id'
})
cm.set_candset_properties(C, 'id', 'l_id', 'r_id', A, B)

True

In [30]:
### ora lo applichiamo a C
brm = em.BooleanRuleMatcher()

brm.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.1',
             'phone_phone_lev_sim(ltuple, rtuple) >= 0.8'], feature_table)


brm.add_rule(['addr_addr_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.1',
              'name_name_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.3', ], feature_table)


predictions = brm.predict(C, target_attr='pred_label',  append=True)

MT=predictions[predictions.pred_label==1]
MT
#
# Porta 'id' come prima colonna
#cols = ['id'] + [col for col in MT.columns if col != 'id']
#MT = MT[cols]

,id,l_id,r_id,l_name,l_addr,l_city,r_name,r_addr,r_city,_sim_score,pred_label
103,103,846,78,motown cafe,104 w. 57th st. near 6th ave.,new york,edison cafe,228 w. 47th st.,new york city,0.324324,1
149,149,808,105,jewel of india,15 w. 44th st.,new york,rose of india,308 e. sixth st.,new york city,0.363636,1
315,315,962,138,binions coffee shop,128 fremont st.,las vegas,steakhouse the,128 e. fremont st.,las vegas,0.382353,1
320,320,974,139,golden nugget hotel,129 e. fremont st.,las vegas,stefanos,129 fremont st.,las vegas,0.426230,1
321,321,976,139,lillie langtrys,129 e. fremont st.,las vegas,stefanos,129 fremont st.,las vegas,0.490909,1
...,...,...,...,...,...,...,...,...,...,...,...
2759,2759,643,328,postrio,545 post st.,san francisco,postrio,545 post st.,san francisco,1.000000,1
2819,2819,644,329,ritz-carlton restaurant and dining room,600 stockton st.,san francisco,ritz-carlton dining room (san francisco),600 stockton st.,san francisco,0.701299,1
2821,2821,645,330,rose pistola,532 columbus ave.,san francisco,rose pistola,532 columbus ave.,san francisco,1.000000,1
2827,2827,623,331,cafe ritz-carlton buckhead,3434 peachtree rd.,atlanta,ritz-carlton cafe (atlanta),181 peachtree st.,atlanta,0.480000,1


In [31]:
Valuta(GoldStandard,MT)

,MT,TP,FP,FN,P,R,F
0,125,108,17,3,0.864,0.973,0.9153


In [32]:
Valuta2(GoldStandard,MT)

,MT,TP,FP,FN,P,R,F
0,125,108,17,3,0.864,0.973,0.9153


# Esempio Matching (Movie)

In questo esempio mostreremo anche come calcolare la similarità dei matching ottenuti

In [33]:
A=pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/wisc_em_benchmark/839_spring19/Movies6/csv/table_a.csv')
B=pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/wisc_em_benchmark/839_spring19/Movies6/csv/table_b.csv')
# ed il relativo gold standard (predicted matches)
GoldStandard=pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/wisc_em_benchmark/839_spring19/Movies6/csv/predicted_matches.csv')
#
CandidateSetDato=pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/wisc_em_benchmark/839_spring19/Movies6/csv/candidate_pairs.csv')
print(A.columns, "\n", B.columns, "\n", GoldStandard.columns , "\n", CandidateSetDato.columns)

Index(['_id', 'id', 'title', 'year', 'genres', 'language', 'runtime', 'budget',
       'revenue', 'directors', 'writers', 'actors'],
      dtype='object') 
 Index(['_id', 'id', 'title', 'year', 'genres', 'language', 'runtime', 'budget',
       'revenue', 'directors', 'writers', 'actors'],
      dtype='object') 
 Index(['id1', 'id2', 'conf'], dtype='object') 
 Index(['id1', 'id2'], dtype='object')


In [34]:
print(GoldStandard.head(2),"\n", GoldStandard.tail(2))

    id1  id2  conf
0     0  171   1.0
1  3465  878   1.0 
       id1   id2  conf
2119  116  1195   0.9
2120  627   359   0.8


In [35]:
# per uniformare i nomi
GoldStandard.columns=['l_id','r_id', 'conf']
CandidateSetDato.columns=['l_id','r_id']
A=A.rename(columns={'_id': 'l_id'})
B=B.rename(columns={'_id': 'r_id'})

# si settano le key dei due dataset
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

GoldStandard['id']=range(len(GoldStandard))
CandidateSetDato['id']=range(len(CandidateSetDato))

cm.set_candset_properties(GoldStandard, 'id', 'l_id', 'r_id', A, B)
cm.set_candset_properties(CandidateSetDato, 'id', 'l_id', 'r_id', A, B)

True

In [36]:
CandidateSetDato.head()

,l_id,r_id,id
0,1001,3171,0
1,1004,183,1
2,1007,3518,2
3,101,607,3
4,1013,894,4


In [37]:
# in fase di match al CandidateSetDato occorre aggiungere gli attributi da usare nel match appunto
CandidateSetDato=em.add_output_attributes(CandidateSetDato,
  l_output_attrs=['title',  'year',  'genres',  'language',  'runtime',  'budget',  'revenue',  'directors',  'writers',  'actors'],
  r_output_attrs=['title',  'year',  'genres',  'language',  'runtime',  'budget',  'revenue',  'directors',  'writers',  'actors'],
								   l_output_prefix='l_', r_output_prefix='r_', delete_from_catalog=False)

In [38]:
CandidateSetDato.head()

,l_id,r_id,id,l_title,l_year,l_genres,l_language,l_runtime,l_budget,l_revenue,...,r_title,r_year,r_genres,r_language,r_runtime,r_budget,r_revenue,r_directors,r_writers,r_actors
0,1001,3171,0,U-571,2000,Action;War,English,116.0,62000000,127666415.0,...,U-571,2000,Action;Drama;Thriller;War,English,116,62000000.0,1.276664e+08,Jonathan Mostow;Cheryl Starbuck;Dianne Dreyer,David Ayer;Sam Montgomery,Matthew McConaughey;Bill Paxton;Harvey Keitel;Jon Bon Jovi;David Keith;Thomas Kretschmann;Jake W...
1,1004,183,1,The Muppet Movie,1979,Adventure;Comedy;Family;Musical,English,95.0,NaN,NaN,...,Harry Potter and the Deathly Hallows: Part 2,2011,Family;Fantasy;Adventure,English,130,125000000.0,1.342000e+09,David Yates,J.K. Rowling;Steve Kloves,Daniel Radcliffe;Rupert Grint;Emma Watson;Alan Rickman;Ralph Fiennes;Maggie Smith;Helena Bonham ...
2,1007,3518,2,Father of the Bride Part II,1995,Comedy;Family;Romance,English,106.0,30000000,NaN,...,Father of the Bride Part II,1995,Comedy,English,106,NaN,7.657891e+07,Charles Shyer,Albert Hackett;Nancy Meyers,Steve Martin;Diane Keaton;Martin Short;Kimberly Williams-Paisley;George Newbern;Kieran Culkin;BD...
3,101,607,3,Monsters University,2013,Animation;Adventure;Comedy;Family;Fantasy,English,104.0,200000000,743559607.0,...,Monsters University,2013,Animation;Family,English,104,200000000.0,7.435596e+08,Dan Scanlon;Russell J. Stough;Matthew Silas;Patrick Lin;Colin Levy;Jan Pfenninger;Gregg Olsson;A...,Allison W. Nelson;Doc Kane;Daniel Gerson;Dan Scanlon;Robert L. Baird;Louis Gonzales,Billy Crystal;John Goodman;Steve Buscemi;Helen Mirren;Noah Johnston;Peter Sohn;Joel Murray;Sean ...
4,1013,894,4,Paddington,2014,Adventure;Comedy;Family;Fantasy,English,95.0,55000000,268047808.0,...,Paddington,2014,Comedy;Adventure;Family,English,96,55000000.0,2.592072e+08,Paul King;Laura Miles,Michael Bond;Paul King;Hamish McColl,Ben Whishaw;Hugh Bonneville;Sally Hawkins;Samuel Joslin;Madeleine Harris;Julie Walters;Nicole Ki...


In [39]:
# Codice completo e generale per ottenere le  feature_table
block_c = em.get_attr_corres(A, B)
atypesA = em.get_attr_types(A)
atypesB = em.get_attr_types(B)
for c in A.columns:
    atypesB[c]=atypesA[c]

feature_table = em.get_features(A, B,
                                 atypesA, atypesB,
                                 block_c,
                                 em.get_tokenizers_for_matching(),
                                 em.get_sim_funs_for_matching())
print(feature_table['feature_name'].to_list())


['id_id_exm', 'id_id_anm', 'id_id_lev_dist', 'id_id_lev_sim', 'title_title_jac_qgm_3_qgm_3', 'title_title_cos_dlm_dc0_dlm_dc0', 'title_title_jac_dlm_dc0_dlm_dc0', 'title_title_mel', 'title_title_lev_dist', 'title_title_lev_sim', 'title_title_nmw', 'title_title_sw', 'year_year_exm', 'year_year_anm', 'year_year_lev_dist', 'year_year_lev_sim', 'genres_genres_lev_dist', 'genres_genres_lev_sim', 'genres_genres_jar', 'genres_genres_jwn', 'genres_genres_exm', 'genres_genres_jac_qgm_3_qgm_3', 'language_language_lev_dist', 'language_language_lev_sim', 'language_language_jar', 'language_language_jwn', 'language_language_exm', 'language_language_jac_qgm_3_qgm_3', 'runtime_runtime_exm', 'runtime_runtime_anm', 'runtime_runtime_lev_dist', 'runtime_runtime_lev_sim', 'budget_budget_jac_qgm_3_qgm_3', 'budget_budget_cos_dlm_dc0_dlm_dc0', 'budget_budget_jac_dlm_dc0_dlm_dc0', 'budget_budget_mel', 'budget_budget_lev_dist', 'budget_budget_lev_sim', 'budget_budget_nmw', 'budget_budget_sw', 'revenue_revenue_e

In [40]:
brm = em.BooleanRuleMatcher()

brm.add_rule(['title_title_lev_sim(ltuple, rtuple) > 0.6'], feature_table)

predictions = brm.predict(CandidateSetDato, target_attr='pred_label',  append=True)
MT=predictions[predictions.pred_label==1]
# Porta 'id' come prima colonna
cols = ['id'] + [col for col in MT.columns if col != 'id']
MT = MT[cols]
MT.sample()

,id,l_id,r_id,l_title,l_year,l_genres,l_language,l_runtime,l_budget,l_revenue,...,r_year,r_genres,r_language,r_runtime,r_budget,r_revenue,r_directors,r_writers,r_actors,pred_label
1279,1279,1875,2361,Meet Joe Black,1998,Drama;Fantasy;Romance,English,178.0,90000000,142940100.0,...,1998,Fantasy;Drama;Mystery,English,178,90000000.0,142940100.0,Martin Brest;Lisa Katcher,Bo Goldman;Kevin Wade;Jeff Reno;Ron Osborn,Brad Pitt;Anthony Hopkins;Claire Forlani;Jake Weber;Marcia Gay Harden;Jeffrey Tambor;June Squibb...,1


In [41]:
#GoldStandard
Valuta(GoldStandard,MT)

,MT,TP,FP,FN,P,R,F
0,2220,2098,122,23,0.945,0.9892,0.9666


In [42]:
Valuta2(GoldStandard,MT)

,MT,TP,FP,FN,P,R,F
0,2220,2098,122,23,0.945,0.9892,0.9666


## Linearly Weighted Matching Rules

In [43]:
brm = em.BooleanRuleMatcher()

brm.add_rule(['title_title_lev_sim(ltuple, rtuple)*0.6 \
              + directors_directors_jac_qgm_3_qgm_3(ltuple, rtuple)*0.2 \
              + actors_actors_cos_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.2 > 0.6'], feature_table)
predictions = brm.predict(CandidateSetDato, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
Valuta2(GoldStandard[['l_id','r_id']],MT[['l_id','r_id']])

,MT,TP,FP,FN,P,R,F
0,2144,2088,56,33,0.9739,0.9844,0.9791


In [44]:
# Valuto qualche FP
VV=VediValuta2(GoldStandard,MT,'FP')
len(VV)

56

In [45]:
pd.merge(pd.merge(VV,A[['l_id','title',  'year',  'genres',  'actors']], on='l_id'),
         B[['r_id','title',  'year',  'genres',  'actors']], on='r_id').head()

,l_id,r_id,_merge,title_x,year_x,genres_x,actors_x,title_y,year_y,genres_y,actors_y
0,1541,3326,right_only,Paranormal Activity 4,2012,Horror;Mystery;Thriller,Katie Featherston;Kathryn Newton;Matt Shively;Aiden Lovekamp;Brady Allen;Stephen Dunham;Alexondr...,Paranormal Activity 3,2011,Horror,Katie Featherston;Sprague Grayden;Lauren Bittner;Christopher Nicholas Smith;Chloe Csengery;Jessi...
1,260,135,right_only,How to Train Your Dragon 2,2014,Animation;Action;Adventure;Comedy;Family;Fantasy,Jay Baruchel;Cate Blanchett;Gerard Butler;Craig Ferguson;America Ferrera;Jonah Hill;Christopher ...,How to Train Your Dragon,2010,Fantasy;Adventure;Animation;Family,Jay Baruchel;Gerard Butler;Craig Ferguson;America Ferrera;Jonah Hill;Christopher Mintz-Plasse;T....
2,314,1920,right_only,Grown Ups,2010,Comedy,Adam Sandler;Kevin James;Chris Rock;David Spade;Rob Schneider;Salma Hayek;Maria Bello;Maya Rudol...,Grown Ups 2,2013,Comedy,Adam Sandler;Kevin James;Chris Rock;David Spade;Salma Hayek;Maya Rudolph;Maria Bello;Nick Swards...
3,503,453,right_only,Rocky III,1982,Drama;Sport,Sylvester Stallone;Talia Shire;Burt Young;Carl Weathers;Burgess Meredith;Tony Burton;Mr. T;Hulk ...,Rocky II,1979,Drama,Sylvester Stallone;Talia Shire;Burt Young;Carl Weathers;Burgess Meredith;Tony Burton;Joe Spinell...
4,824,3518,right_only,Father of the Bride,1991,Comedy;Family;Romance,Steve Martin;Diane Keaton;Kimberly Williams-Paisley;Kieran Culkin;George Newbern;Martin Short;BD...,Father of the Bride Part II,1995,Comedy,Steve Martin;Diane Keaton;Martin Short;Kimberly Williams-Paisley;George Newbern;Kieran Culkin;BD...


In [46]:
# I FP sembrano dovuti ad anno differente,
# Ricordiamo che
# Each rule is a conjunction of predicates.
# That is, each rule will return True only if all the predicates return True.
# Allora metteremo in AND year_year_exm

#proviamo prima singolarmente
# 'year_year_exm(ltuple, rtuple) != 0'
brm = em.BooleanRuleMatcher()
brm.add_rule(['year_year_exm(ltuple, rtuple) != 0'], feature_table)
predictions = brm.predict(CandidateSetDato, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
MT[['l_year','r_year']].sample(5)


,l_year,r_year
365,2009,2009
2931,2003,2003
497,2018,2018
131,1991,1991
1357,2002,2002


In [47]:
# Quindi mettiamo in AND


In [48]:
brm = em.BooleanRuleMatcher()
brm.add_rule(['title_title_lev_sim(ltuple, rtuple)*0.6 \
                + directors_directors_jac_qgm_3_qgm_3(ltuple, rtuple)*0.2 \
                  + actors_actors_cos_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.2 > 0.6'
              
              , 'year_year_exm(ltuple, rtuple) != 0'
              ], feature_table)
predictions = brm.predict(CandidateSetDato, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
Valuta2(GoldStandard[['l_id','r_id']],MT[['l_id','r_id']])

,MT,TP,FP,FN,P,R,F
0,2074,2074,0,47,1.0,0.9778,0.9888


In [49]:
# provo a ridurre soglia
brm = em.BooleanRuleMatcher()
brm.add_rule(['title_title_lev_sim(ltuple, rtuple)*0.6 + directors_directors_jac_qgm_3_qgm_3(ltuple, rtuple)*0.2 \
              + actors_actors_cos_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.2 > 0.25', 'year_year_exm(ltuple, rtuple) != 0'], feature_table)
predictions = brm.predict(CandidateSetDato, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
Valuta2(GoldStandard[['l_id','r_id']],MT[['l_id','r_id']])

,MT,TP,FP,FN,P,R,F
0,2103,2102,1,19,0.9995,0.991,0.9953


In [50]:
# supponniamo di usare 
MT

,l_id,r_id,id,l_title,l_year,l_genres,l_language,l_runtime,l_budget,l_revenue,...,r_year,r_genres,r_language,r_runtime,r_budget,r_revenue,r_directors,r_writers,r_actors,pred_label
0,1001,3171,0,U-571,2000,Action;War,English,116.0,62000000,127666415.0,...,2000,Action;Drama;Thriller;War,English,116,62000000.0,127666415.0,Jonathan Mostow;Cheryl Starbuck;Dianne Dreyer,David Ayer;Sam Montgomery,Matthew McConaughey;Bill Paxton;Harvey Keitel;Jon Bon Jovi;David Keith;Thomas Kretschmann;Jake W...,1
2,1007,3518,2,Father of the Bride Part II,1995,Comedy;Family;Romance,English,106.0,30000000,NaN,...,1995,Comedy,English,106,NaN,76578911.0,Charles Shyer,Albert Hackett;Nancy Meyers,Steve Martin;Diane Keaton;Martin Short;Kimberly Williams-Paisley;George Newbern;Kieran Culkin;BD...,1
3,101,607,3,Monsters University,2013,Animation;Adventure;Comedy;Family;Fantasy,English,104.0,200000000,743559607.0,...,2013,Animation;Family,English,104,200000000.0,743559607.0,Dan Scanlon;Russell J. Stough;Matthew Silas;Patrick Lin;Colin Levy;Jan Pfenninger;Gregg Olsson;A...,Allison W. Nelson;Doc Kane;Daniel Gerson;Dan Scanlon;Robert L. Baird;Louis Gonzales,Billy Crystal;John Goodman;Steve Buscemi;Helen Mirren;Noah Johnston;Peter Sohn;Joel Murray;Sean ...,1
4,1013,894,4,Paddington,2014,Adventure;Comedy;Family;Fantasy,English,95.0,55000000,268047808.0,...,2014,Comedy;Adventure;Family,English,96,55000000.0,259207227.0,Paul King;Laura Miles,Michael Bond;Paul King;Hamish McColl,Ben Whishaw;Hugh Bonneville;Sally Hawkins;Samuel Joslin;Madeleine Harris;Julie Walters;Nicole Ki...,1
5,1022,1603,5,The Jewel of the Nile,1985,Action;Adventure;Comedy;Romance,English,106.0,25000000,96773200.0,...,1985,Action;Adventure;Comedy;Romance,English,106,25000000.0,96773200.0,Lewis Teague,Diane Thomas;Lawrence Konner;Mark Rosenthal,Michael Douglas;Kathleen Turner;Danny DeVito;Holland Taylor;Spiros Focs;Avner Eisenberg;Paul Dav...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3082,3735,2187,3082,Silent Hill: Revelation,2012,Horror,English,95.0,20000000,52302796.0,...,2012,Thriller;Horror;Mystery,English,94,20000000.0,52302796.0,M.J. Bassett;Lisa Burling,Laurent Hadida;Masahiro It;M.J. Bassett,Adelaide Clemens;Radha Mitchell;Carrie-Anne Moss;Sean Bean;Malcolm McDowell;Kit Harington;Debora...,1
3116,3888,2170,3116,Elizabeth: The Golden Age,2007,Biography;Drama;History;War,English,114.0,55000000,74870866.0,...,2007,Drama;History;Romance,English,114,55000000.0,74237563.0,Shekhar Kapur;Angela Wharton,Michael Hirst;William Nicholson,Cate Blanchett;Clive Owen;Geoffrey Rush;Laurence Fox;Tom Hollander;Abbie Cornish;Rhys Ifans;Jord...,1
3145,1312,2892,3145,The Wolfman,2010,Drama;Fantasy;Horror;Thriller,English,103.0,150000000,139789765.0,...,2010,Drama;Horror;Thriller;Fantasy,English,102,150000000.0,NaN,Joe Johnston;Charlotte Keating;Lisa Vick;Katie Harlow,Curt Siodmak;David Self;Andrew Kevin Walker,Benicio del Toro;Anthony Hopkins;Emily Blunt;Hugo Weaving;Cristina Contes;David Sterne;Elizabeth...,1
3151,1223,3306,3151,The Manchurian Candidate,2004,Drama;Mystery;Sci-Fi;Thriller,English,129.0,80000000,96105964.0,...,2004,Drama;Thriller;Mystery,English,129,80000000.0,96105964.0,Betsy Friedman;Arthur Anderson;Jonathan Demme;Eva Z. Cabrera;Maggie Murphy;Matthew Sirianni;Doug...,Richard Condon;Dean Georgaris;Daniel Pyne;Mary Tobler;Narges Takesh;Kalina Ivanov,Denzel Washington;Liev Schreiber;Meryl Streep;Kimberly Elise;Jon Voight;Vera Farmiga;Bruno Ganz;...,1


Supponiamo di fermarci qui, e considerare questa come la Match Table
finale.
Si calcolo quindi la similarità tra le coppie valutando


```
'title_title_lev_sim(ltuple, rtuple)*0.6 + directors_directors_jac_qgm_3_qgm_3(ltuple, rtuple)*0.2 \
              + actors_actors_cos_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.2```



In [51]:
MT['id']=range(len(MT))
cm.set_candset_properties(MT, 'id', 'l_id', 'r_id', A, B)

True

In [52]:
# considero solo le features intressate dalla rule
RuleFeatureList=['directors_directors_jac_qgm_3_qgm_3',
                 'actors_actors_cos_dlm_dc0_dlm_dc0',
                 'title_title_lev_sim']

RuleFeatureTable=feature_table[feature_table['feature_name'].isin(RuleFeatureList)]

#RuleFeatureVector


FeaturesVector=em.extract_feature_vecs(MT, feature_table=RuleFeatureTable)
FeaturesVector

,id,l_id,r_id,title_title_lev_sim,directors_directors_jac_qgm_3_qgm_3,actors_actors_cos_dlm_dc0_dlm_dc0
0,0,1001,3171,1.000000,0.306122,0.512272
2,1,1007,3518,1.000000,1.000000,0.404061
3,2,101,607,1.000000,0.049793,0.240287
4,3,1013,894,1.000000,0.360000,0.223825
5,4,1022,1603,1.000000,1.000000,0.546401
...,...,...,...,...,...,...
3082,2098,3735,2187,0.884615,0.413793,0.182574
3116,2099,3888,2170,1.000000,0.419355,0.024746
3145,2100,1312,2892,1.000000,0.222222,0.024746
3151,2101,1223,3306,1.000000,0.111111,0.043519


In [53]:
MT=em.extract_feature_vecs(MT, feature_table=RuleFeatureTable)

In [54]:
MT

,id,l_id,r_id,title_title_lev_sim,directors_directors_jac_qgm_3_qgm_3,actors_actors_cos_dlm_dc0_dlm_dc0
0,0,1001,3171,1.000000,0.306122,0.512272
2,1,1007,3518,1.000000,1.000000,0.404061
3,2,101,607,1.000000,0.049793,0.240287
4,3,1013,894,1.000000,0.360000,0.223825
5,4,1022,1603,1.000000,1.000000,0.546401
...,...,...,...,...,...,...
3082,2098,3735,2187,0.884615,0.413793,0.182574
3116,2099,3888,2170,1.000000,0.419355,0.024746
3145,2100,1312,2892,1.000000,0.222222,0.024746
3151,2101,1223,3306,1.000000,0.111111,0.043519


In [55]:
MT['sim']= MT['title_title_lev_sim']*0.6 \
            + MT['directors_directors_jac_qgm_3_qgm_3']*0.2  \
             +MT['actors_actors_cos_dlm_dc0_dlm_dc0']*0.2
MT.sort_values('sim')

,id,l_id,r_id,title_title_lev_sim,directors_directors_jac_qgm_3_qgm_3,actors_actors_cos_dlm_dc0_dlm_dc0,sim
1298,939,1977,339,0.148936,0.347826,0.481189,0.255165
2095,1514,1438,3619,0.454545,0.000000,0.000000,0.272727
544,389,2819,2900,0.441176,0.233333,0.140642,0.339501
73,54,1226,1946,0.300000,0.413793,0.401643,0.343087
1951,1406,10,920,0.321429,0.424242,0.346844,0.347074
...,...,...,...,...,...,...,...
2408,1738,2641,2804,1.000000,1.000000,0.930484,0.986097
1468,1064,2631,3158,1.000000,1.000000,0.965247,0.993049
2945,2072,976,247,1.000000,1.000000,1.000000,1.000000
887,625,692,736,1.000000,1.000000,1.000000,1.000000


In [56]:
MT.columns

Index(['id', 'l_id', 'r_id', 'title_title_lev_sim',
       'directors_directors_jac_qgm_3_qgm_3',
       'actors_actors_cos_dlm_dc0_dlm_dc0', 'sim'],
      dtype='object')

In [57]:
MTsm=stable_marriage(MT[['l_id', 'r_id', "sim"]])


In [58]:
Valuta2(GoldStandard[['l_id','r_id']],MT[['l_id','r_id']])

,MT,TP,FP,FN,P,R,F
0,2103,2102,1,19,0.9995,0.991,0.9953


In [59]:
Valuta2(GoldStandard[['l_id','r_id']],MTsm[['l_id','r_id']])

,MT,TP,FP,FN,P,R,F
0,2101,2101,0,20,1.0,0.9906,0.9953


# Esempio Clean (No Duplicati)

In [60]:
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/Sintetici/'
src_links = [
path+'S1_clean_.csv',
path+'S2_clean_.csv',
path+'S3_clean_.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

SOURCES['S3']=SOURCES['S3'][SOURCES['S3'].id!='S3_2']
GoldStandardCLEAN=pd.read_csv(path+"GoldStandardClean.csv")
GoldStandardCLEAN.sample()

,l_id,r_id,sim
3,S1_6,S2_0,0.53125


In [61]:
SOURCES['S3']

,given_name,surname,date_of_birth,id
0,emmerson,loyck,19211129,S3_0
1,michel,wuchatsch,19190110,S3_1
2,liersch,michael,19360816,S3_3
3,charlotte,hyland,19460401,S3_4
4,braedon,schuetz,19440909,S3_5
5,olivia,hobson,19760812,S3_6
6,joshua,green,19790110,S3_7
7,keely,clarke,19050410,S3_8
8,joshua,morriosn,19101123,S3_9
9,genovefa,hyllande,19071008,S3_11


In [62]:
# IdSOURCES(SOURCES) fornisce tutti i nodi del grafo da usare in ClusterComponentiConnessi
TuttiInodi=IdSOURCES(SOURCES)
print(TuttiInodi)


['S1_0', 'S1_1', 'S1_2', 'S1_3', 'S1_4', 'S1_5', 'S1_6', 'S1_7', 'S1_8', 'S1_9', 'S1_10', 'S1_11', 'S2_0', 'S2_1', 'S2_2', 'S2_3', 'S2_4', 'S2_5', 'S2_6', 'S2_7', 'S3_0', 'S3_1', 'S3_3', 'S3_4', 'S3_5', 'S3_6', 'S3_7', 'S3_8', 'S3_9', 'S3_11']


In [63]:
GoldStandardCLEAN

,l_id,r_id,sim
0,S1_8,S2_4,1.000000
1,S1_9,S2_2,0.793103
2,S1_10,S2_7,0.540541
3,S1_6,S2_0,0.531250
4,S1_3,S2_1,0.526316
5,S1_5,S3_0,1.000000
6,S1_8,S3_6,1.000000
7,S1_4,S3_1,0.862069
8,S1_7,S3_7,0.807692
9,S1_0,S3_9,0.733333


In [64]:
ClusterGoldStandardCLEAN=ClusterComponentiConnessi(GoldStandardCLEAN[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
ClusterGoldStandardCLEAN

,ClusterKey,ClusterElement
0,1,S2_4
1,1,S1_8
2,1,S3_6
3,2,S3_3
4,2,S2_2
5,2,S1_9
6,3,S1_10
7,3,S2_7
8,4,S2_0
9,4,S3_5


In [65]:
# per  visualizzare i cluster, si raggruppa sulla ClusterKey
def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:2]

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
ClusterGoldStandardCLEAN.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Sources', ascending=False)

,ClusterKey,#Sources,Sources,#Elements,Elements
0,1,3,"S2,S1,S3",3,"S2_4,S1_8,S3_6"
1,2,3,"S3,S2,S1",3,"S3_3,S2_2,S1_9"
3,4,3,"S2,S3,S1",3,"S2_0,S3_5,S1_6"
7,8,3,"S1,S3,S2",3,"S1_7,S3_7,S2_5"
2,3,2,"S1,S2",2,"S1_10,S2_7"
4,5,2,"S2,S1",2,"S2_1,S1_3"
5,6,2,"S1,S3",2,"S1_5,S3_0"
6,7,2,"S1,S3",2,"S1_4,S3_1"
8,9,2,"S3,S1",2,"S3_9,S1_0"
9,10,2,"S3,S1",2,"S3_11,S1_11"


In [66]:
# questo è riportato nella VisualizzaCluster; è necessario che la sorgente sia indicata come il primo, 
#o i primi due caratteri
#       x['source']=x['ClusterElement'].astype(str).str[:2]

def VisualizzaCluster(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[:2]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS

VisualizzaCluster(ClusterGoldStandardCLEAN)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,3,"S2,S1,S3",3,"S2_4,S1_8,S3_6"
1,2,3,"S3,S2,S1",3,"S3_3,S2_2,S1_9"
2,3,2,"S1,S2",2,"S1_10,S2_7"
3,4,3,"S2,S3,S1",3,"S2_0,S3_5,S1_6"
4,5,2,"S2,S1",2,"S2_1,S1_3"
5,6,2,"S1,S3",2,"S1_5,S3_0"
6,7,2,"S1,S3",2,"S1_4,S3_1"
7,8,3,"S1,S3,S2",3,"S1_7,S3_7,S2_5"
8,9,2,"S3,S1",2,"S3_9,S1_0"
9,10,2,"S3,S1",2,"S3_11,S1_11"


In [67]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)

Numero Elementi 30
Cluster con max numero di elementi: [1, 2, 4, 8]


,NumeroElementiPerCluster,NumeroCluster
0,1,4
1,2,7
2,3,4


In [68]:
# GoldStandardCLEAN è riferito a tutte e tre le sorgenti, e contiene solo le coppie  Si_, Sj, con i < j
GoldStandardCLEAN.sample(5)

,l_id,r_id,sim
4,S1_3,S2_1,0.526316
16,S2_6,S3_4,0.542857
9,S1_0,S3_9,0.733333
6,S1_8,S3_6,1.000000
0,S1_8,S2_4,1.000000


In [69]:
# per ottenere il gold standard relativo ad   una coppia di sorgenti
def GoldStandard(GS,s1,s2):
    return GS[ (GS['l_id'].isin(SOURCES[s1]['id'])) & GS['r_id'].isin(SOURCES[s2]['id'])]

In [70]:
GoldStandard(GoldStandardCLEAN,'S1','S3')

,l_id,r_id,sim
5,S1_5,S3_0,1.000000
6,S1_8,S3_6,1.000000
7,S1_4,S3_1,0.862069
8,S1_7,S3_7,0.807692
9,S1_0,S3_9,0.733333
10,S1_11,S3_11,0.676471
11,S1_6,S3_5,0.531250
12,S1_9,S3_3,0.529412


## Matching tra una coppia di sources

In [71]:
# si considera la coppia di sources
A=SOURCES['S1']
B=SOURCES['S3']

A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })

em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

True

In [72]:
# BLOCKING

A['mix'] = A['given_name'] + ' ' + A['surname'] + ' ' + A['date_of_birth']
B['mix'] = B['given_name'] + ' ' + B['surname'] + ' ' + B['date_of_birth']

C_SimJoin_mix  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                    'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                    l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                    r_out_attrs=['given_name', 'surname', 'date_of_birth'])
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'l_l_id': 'l_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'r_r_id': 'r_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'_sim_score': 'sim'})

em.set_key(C_SimJoin_mix, '_id')
em.set_ltable(C_SimJoin_mix, A)
em.set_rtable(C_SimJoin_mix, B)
em.set_fk_ltable(C_SimJoin_mix, 'l_id')
em.set_fk_rtable(C_SimJoin_mix, 'r_id')
C_SimJoin_mix

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069
3,3,S1_9,S3_3,michael,lierach,19360816,liersch,michael,19360816,0.529412
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471


In [73]:
C_SimJoin_mix.columns
GS=GoldStandard(GoldStandardCLEAN,'S1','S3')
GS.columns

Index(['l_id', 'r_id', 'sim'], dtype='object')

In [74]:
ValutaBlocking(A,B,C_SimJoin_mix,GoldStandard(GoldStandardCLEAN,'S1','S3'))

,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,12,10,9,0.925,1.0,0.8889


In [75]:
# Matching

In [76]:
# Codice completo e generale per ottenere le  feature_table
block_c = em.get_attr_corres(A, B)
atypesA = em.get_attr_types(A)
atypesB = em.get_attr_types(B)
for c in A.columns:
    atypesB[c]=atypesA[c]

feature_table = em.get_features(A, B,
                                 atypesA, atypesB,
                                 block_c,
                                 em.get_tokenizers_for_matching(),
                                 em.get_sim_funs_for_matching())
print(feature_table['feature_name'].to_list())

['given_name_given_name_lev_dist', 'given_name_given_name_lev_sim', 'given_name_given_name_jar', 'given_name_given_name_jwn', 'given_name_given_name_exm', 'given_name_given_name_jac_qgm_3_qgm_3', 'surname_surname_lev_dist', 'surname_surname_lev_sim', 'surname_surname_jar', 'surname_surname_jwn', 'surname_surname_exm', 'surname_surname_jac_qgm_3_qgm_3', 'date_of_birth_date_of_birth_lev_dist', 'date_of_birth_date_of_birth_lev_sim', 'date_of_birth_date_of_birth_jar', 'date_of_birth_date_of_birth_jwn', 'date_of_birth_date_of_birth_exm', 'date_of_birth_date_of_birth_jac_qgm_3_qgm_3', 'mix_mix_jac_qgm_3_qgm_3', 'mix_mix_cos_dlm_dc0_dlm_dc0', 'mix_mix_jac_dlm_dc0_dlm_dc0', 'mix_mix_mel', 'mix_mix_lev_dist', 'mix_mix_lev_sim', 'mix_mix_nmw', 'mix_mix_sw']


In [77]:
# in questi esempi simbolici usiamo direttamente
feature_table = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

In [78]:
feature_table

,feature_name,left_attribute,right_attribute,left_attr_tokenizer,right_attr_tokenizer,simfunction,function,function_source,is_auto_generated
0,given_name_given_name_lev_dist,given_name,given_name,None,None,lev_dist,<function given_name_given_name_lev_dist at 0x7d04af5404d0>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
1,given_name_given_name_lev_sim,given_name,given_name,None,None,lev_sim,<function given_name_given_name_lev_sim at 0x7d04af540680>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
2,given_name_given_name_jar,given_name,given_name,None,None,jaro,<function given_name_given_name_jar at 0x7d04af5407a0>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
3,given_name_given_name_jwn,given_name,given_name,None,None,jaro_winkler,<function given_name_given_name_jwn at 0x7d04af5408c0>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
4,given_name_given_name_exm,given_name,given_name,None,None,exact_match,<function given_name_given_name_exm at 0x7d04af5409e0>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
5,given_name_given_name_jac_qgm_3_qgm_3,given_name,given_name,qgm_3,qgm_3,jaccard,<function given_name_given_name_jac_qgm_3_qgm_3 at 0x7d04af540b00>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
6,surname_surname_lev_dist,surname,surname,None,None,lev_dist,<function surname_surname_lev_dist at 0x7d04af540c20>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
7,surname_surname_lev_sim,surname,surname,None,None,lev_sim,<function surname_surname_lev_sim at 0x7d04af540d40>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
8,surname_surname_jar,surname,surname,None,None,jaro,<function surname_surname_jar at 0x7d04af540e60>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True
9,surname_surname_jwn,surname,surname,None,None,jaro_winkler,<function surname_surname_jwn at 0x7d04af540f80>,from py_entitymatching.feature.simfunctions import *\nfrom py_entitymatching.feature.tokenizers ...,True


In [79]:
# DOMANDA: commentare il significato della seguente rule e valutarne il risultato

brm = em.BooleanRuleMatcher()
brm.add_rule(['date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1',
              ], feature_table) # stessa entita se stessa data di nascita 

# OR

brm.add_rule(['surname_surname_lev_sim(ltuple, rtuple)*0.5 \
                + given_name_given_name_lev_sim(ltuple, rtuple)*0.2 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.7'], feature_table)
predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
MT

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069,1
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250,1
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000,1
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692,1
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333,1
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,1


In [80]:
Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)

,MT,TP,FP,FN,P,R,F
0,8,7,1,1,0.875,0.875,0.875


In [81]:
#Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3')[['l_id','r_id']],MT[['l_id','r_id']])

In [82]:
VV=VediValuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FP')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,given_name_y,surname_y,date_of_birth_y,mix_y
0,S1_2,S3_0,right_only,emmerson,lock,19211129,emmerson lock 19211129,emmerson,loyck,19211129,emmerson loyck 19211129


In [83]:
VV=VediValuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FN')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,given_name_y,surname_y,date_of_birth_y,mix_y
0,S1_9,S3_3,left_only,michael,lierach,19360816,michael lierach 19360816,liersch,michael,19360816,liersch michael 19360816


In [84]:
# come considerare questo falso negativo dovuto allo scambio tra nome e cognome?
# definire NAME come concatenazione ed usare una jaccard
# ()

In [85]:
# non è necessario rifare blocking, ma solo definire il nuovo attributo
A['NAME'] = A['given_name'] + ' ' + A['surname']
B['NAME'] = B['given_name'] + ' ' + B['surname']

# e rigenerare le features, verificando le features per il nuovo attributo
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)


In [86]:
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
brm = em.BooleanRuleMatcher()
brm.add_rule(['surname_surname_exm(ltuple, rtuple) == 1', 'date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1'], F)
brm.add_rule(['NAME_NAME_jac_qgm_3_qgm_3(ltuple, rtuple)*0.6 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.4 > 0.5'], F)
predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
MT

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069,1
3,3,S1_9,S3_3,michael,lierach,19360816,liersch,michael,19360816,0.529412,1
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250,1
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000,1
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692,1
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333,1
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,1


In [87]:
Valuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)

,MT,TP,FP,FN,P,R,F
0,9,8,1,0,0.8889,1.0,0.9412


In [88]:
VV=VediValuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FP')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,NAME_x,given_name_y,surname_y,date_of_birth_y,mix_y,NAME_y
0,S1_2,S3_0,right_only,emmerson,lock,19211129,emmerson lock 19211129,emmerson lock,emmerson,loyck,19211129,emmerson loyck 19211129,emmerson loyck


In [89]:
# 

In [90]:
# 

In [91]:
GS=GoldStandard(GoldStandardCLEAN,'S1','S3')
# Uno-a-molti (l_id → più r_id)
GS[GS.duplicated('l_id', keep=False)]

,l_id,r_id,sim


In [92]:
# Molti-a-uno (r_id → più l_id)
GS[GS.duplicated('r_id', keep=False)]

,l_id,r_id,sim


In [93]:
MT[MT.duplicated('r_id', keep=False)]

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1


In [94]:
MT[MT.duplicated('l_id', keep=False)]

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label


In [95]:
MT=stable_marriage(MT)

In [96]:
MT
# Porta 'id' come prima colonna
cols = ['_id'] + [col for col in MT.columns if col != '_id']
MT = MT[cols]
MT


,_id,l_id,r_id,sim,l_date_of_birth,l_given_name,l_surname,pred_label,r_date_of_birth,r_given_name,r_surname
0,1.0,S1_5,S3_0,1.000000,19211129,emmerson,loyck,1.0,19211129,emmerson,loyck
1,5.0,S1_8,S3_6,1.000000,19760812,olivia,hobson,1.0,19760812,olivia,hobson
2,2.0,S1_4,S3_1,0.862069,19190110,michael,wuchatsch,1.0,19190110,michel,wuchatsch
3,6.0,S1_7,S3_7,0.807692,19790110,joshua,greenj,1.0,19790110,joshua,green
4,7.0,S1_0,S3_9,0.733333,19101123,joshua,morrison,1.0,19101123,joshua,morriosn
5,8.0,S1_11,S3_11,0.676471,19071008,genoveffa,hylander,1.0,19071008,genovefa,hyllande
6,4.0,S1_6,S3_5,0.531250,19440909,rhys,schuetz,1.0,19440909,braedon,schuetz
7,3.0,S1_9,S3_3,0.529412,19360816,michael,lierach,1.0,19360816,liersch,michael


In [97]:
Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)

,MT,TP,FP,FN,P,R,F
0,8,8,0,0,1.0,1.0,1.0


# Funzione BlockingMatchingRule(A,B):

Riassumiamo tutto in una singola funzione

In [98]:
def BlockingMatchingRule(A,B):
  A=A.rename(columns={'id': "l_id" })
  B=B.rename(columns={'id': "r_id" })

  A['NAME'] = A['given_name'] + ' ' + A['surname']
  B['NAME'] = B['given_name'] + ' ' + B['surname']

  A['mix'] = A['given_name'] + ' ' + A['surname'] + ' ' + A['date_of_birth']
  B['mix'] = B['given_name'] + ' ' + B['surname'] + ' ' + B['date_of_birth']

  em.set_key(A, 'l_id')
  em.set_key(B, 'r_id')

# BLOCKING
  C_SimJoin_mix  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                    'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                    l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                    r_out_attrs=['given_name', 'surname', 'date_of_birth'])
  C_SimJoin_mix=C_SimJoin_mix.rename(columns={'l_l_id': 'l_id'})
  C_SimJoin_mix=C_SimJoin_mix.rename(columns={'r_r_id': 'r_id'})
  C_SimJoin_mix=C_SimJoin_mix.rename(columns={'_sim_score': 'sim'})

  em.set_key(C_SimJoin_mix, '_id')
  em.set_ltable(C_SimJoin_mix, A)
  em.set_rtable(C_SimJoin_mix, B)
  em.set_fk_ltable(C_SimJoin_mix, 'l_id')
  em.set_fk_rtable(C_SimJoin_mix, 'r_id')
  em.set_fk_rtable(C_SimJoin_mix, 'r_id')

# MATCHING

  F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
  brm = em.BooleanRuleMatcher()

  brm.add_rule(['surname_surname_exm(ltuple, rtuple) == 1',
                'date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1'], F)

  brm.add_rule(['NAME_NAME_jac_qgm_3_qgm_3(ltuple, rtuple)*0.6 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.4 > 0.5'], F)
  predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
  MT=predictions[predictions.pred_label==1]

  return MT


# Matching tra n sorgenti

La funzione BlockingMatchingRule che viene applicata a tutte le coppie di sorgenti


In [99]:
def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [100]:
MatchTableSOURCES(SOURCES)

,l_id,r_id,sim
0,S1_8,S2_4,1.000000
1,S1_9,S2_2,0.793103
2,S1_10,S2_7,0.540541
3,S1_6,S2_0,0.531250
4,S1_3,S2_1,0.526316
5,S1_5,S2_3,0.500000
0,S1_5,S3_0,1.000000
1,S1_8,S3_6,1.000000
2,S1_4,S3_1,0.862069
3,S1_7,S3_7,0.807692


In [101]:
# clusterizzazione
MTSOURCES=MatchTableSOURCES(SOURCES)

CLUSTERS=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
CLUSTERS

,ClusterKey,ClusterElement
0,1,S2_4
1,1,S1_8
2,1,S3_6
3,2,S3_3
4,2,S2_2
5,2,S1_9
6,3,S1_10
7,3,S2_7
8,4,S2_0
9,4,S3_5


In [102]:
VisualizzaCluster(CLUSTERS).sort_values('#Sorgenti', ascending=False)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,3,"S2,S1,S3",3,"S2_4,S1_8,S3_6"
1,2,3,"S3,S2,S1",3,"S3_3,S2_2,S1_9"
3,4,3,"S2,S3,S1",3,"S2_0,S3_5,S1_6"
5,6,3,"S1,S3,S2",3,"S1_5,S3_0,S2_3"
7,8,3,"S1,S3,S2",3,"S1_7,S3_7,S2_5"
2,3,2,"S1,S2",2,"S1_10,S2_7"
4,5,2,"S2,S1",2,"S2_1,S1_3"
6,7,2,"S1,S3",2,"S1_4,S3_1"
8,9,2,"S3,S1",2,"S3_9,S1_0"
9,10,2,"S3,S1",2,"S3_11,S1_11"


In [103]:
_VisualizzaDistribuzioneCluster(CLUSTERS)

Numero Elementi 30
Cluster con max numero di elementi: [1, 2, 4, 6, 8]


,NumeroElementiPerCluster,NumeroCluster
0,1,3
1,2,6
2,3,5


In [104]:
# confrontiamola con quella della clusterizzazione derivata dal Gold Standard
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)

Numero Elementi 30
Cluster con max numero di elementi: [1, 2, 4, 8]


,NumeroElementiPerCluster,NumeroCluster
0,1,4
1,2,7
2,3,4


In [105]:
VisualizzaCluster(ClusterGoldStandardCLEAN).sort_values('#Sorgenti', ascending=False)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,3,"S2,S1,S3",3,"S2_4,S1_8,S3_6"
1,2,3,"S3,S2,S1",3,"S3_3,S2_2,S1_9"
3,4,3,"S2,S3,S1",3,"S2_0,S3_5,S1_6"
7,8,3,"S1,S3,S2",3,"S1_7,S3_7,S2_5"
2,3,2,"S1,S2",2,"S1_10,S2_7"
4,5,2,"S2,S1",2,"S2_1,S1_3"
5,6,2,"S1,S3",2,"S1_5,S3_0"
6,7,2,"S1,S3",2,"S1_4,S3_1"
8,9,2,"S3,S1",2,"S3_9,S1_0"
9,10,2,"S3,S1",2,"S3_11,S1_11"


In [106]:
## Nei ClusterGoldStandardCLEAN c'è un singleton in più, relativo a S2_3, che non c'è nel calcolo effettuato
# questo significa   che la nostra clusterizzazione ha accoppiato S2_3 ... quindi ha generato uno o più falsi positivi

# Questo si può vedere tramite i match indotti
# valutando cioè i match indotti dal gold standard e quelli dal calcolo da noi effettuato

In [107]:
CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN)
CalcolaMatchIndottiCluster(CLUSTERS)

,l_id,r_id
2,S2_4,S3_6
3,S1_8,S2_4
5,S1_8,S3_6
12,S2_2,S3_3
15,S1_9,S3_3
16,S1_9,S2_2
19,S1_10,S2_7
23,S2_0,S3_5
28,S1_6,S2_0
29,S1_6,S3_5


In [108]:
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(CLUSTERS))

,MT,TP,FP,FN,P,R,F
0,21,19,2,0,0.9048,1.0,0.95


In [109]:
VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(CLUSTERS), 'FP')

,l_id,r_id,_merge
19,S1_5,S2_3,right_only
20,S2_3,S3_0,right_only


In [110]:
# vediamo i record corrispondenti nelle sources

In [111]:
SOURCES['S1'][SOURCES['S1'].id=='S1_5']

,given_name,surname,date_of_birth,id
5,emmerson,loyck,19211129,S1_5


In [112]:
SOURCES['S2'][SOURCES['S2'].id=='S2_3']

,given_name,surname,date_of_birth,id
3,emmeron,loyk,19321129,S2_3


In [113]:
SOURCES['S3'][SOURCES['S3'].id=='S3_0']

,given_name,surname,date_of_birth,id
0,emmerson,loyck,19211129,S3_0


# Esempio 3 sorgenti clean

In [114]:
# CASO CLEAN
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/NCVR2/'

src_links = [
path+'NCVR_AF_clean.csv',
path+'NCVR_BF_clean.csv',
path+'NCVR_CF_clean.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandardCLEAN=pd.read_csv(path + "GoldStandardCLEAN2.csv")

In [115]:
ClusterGoldStandardCLEAN=ClusterComponentiConnessi(GoldStandardCLEAN[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
ClusterGoldStandardCLEAN.sample()

,ClusterKey,ClusterElement
250,162,1_5040_6886105


In [116]:
VisualizzaCluster(ClusterGoldStandardCLEAN).sort_values('#Sorgenti', ascending=False).head()

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
27,28,3,"0_,2_,1_",3,"0_1240_1391987,2_1240_13627651,1_1240_1340473"
3,4,3,"2_,0_,1_",3,"2_140_12924542,0_140_9704280,1_140_9350108"
4,5,3,"2_,0_,1_",3,"2_222_12748029,0_222_3198122,1_222_13226442"
5,6,3,"0_,1_,2_",3,"0_240_94472,1_240_116265,2_240_13265262"
21,22,3,"1_,0_,2_",3,"1_1022_8529932,0_1022_7968679,2_1022_7999709"


In [117]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)

Numero Elementi 338
Cluster con max numero di elementi: [4, 5, 6, 19, 22, 23, 27, 28]


,NumeroElementiPerCluster,NumeroCluster
0,1,158
1,2,78
2,3,8


###Premessa

Con il seguente codice si fissano e si verificano le features da usare nel matching

In [118]:
# Consideriamo la loro unione nel dataframe UNIONE
# il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
# quindi effettuo unione tramite append
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
FeaturesList

['first_name_first_name_jac_qgm_3_qgm_3',
 'first_name_first_name_cos_dlm_dc0_dlm_dc0',
 'first_name_first_name_jac_dlm_dc0_dlm_dc0',
 'first_name_first_name_mel',
 'first_name_first_name_lev_dist',
 'first_name_first_name_lev_sim',
 'first_name_first_name_nmw',
 'first_name_first_name_sw',
 'last_name_last_name_jac_qgm_3_qgm_3',
 'last_name_last_name_cos_dlm_dc0_dlm_dc0',
 'last_name_last_name_jac_dlm_dc0_dlm_dc0',
 'last_name_last_name_mel',
 'last_name_last_name_lev_dist',
 'last_name_last_name_lev_sim',
 'last_name_last_name_nmw',
 'last_name_last_name_sw',
 'sex_sex_lev_dist',
 'sex_sex_lev_sim',
 'sex_sex_jar',
 'sex_sex_jwn',
 'sex_sex_exm',
 'sex_sex_jac_qgm_3_qgm_3',
 'age_age_lev_dist',
 'age_age_lev_sim',
 'age_age_jar',
 'age_age_jwn',
 'age_age_exm',
 'age_age_jac_qgm_3_qgm_3',
 'birth_place_birth_place_jac_qgm_3_qgm_3',
 'birth_place_birth_place_cos_dlm_dc0_dlm_dc0',
 'birth_place_birth_place_jac_dlm_dc0_dlm_dc0',
 'birth_place_birth_place_mel',
 'birth_place_birth_place_

In [119]:
# Features da considerare nel matching
FixedFeatures = F[F.feature_name.isin(['last_name_last_name_lev_sim',
                                       'zip_code_zip_code_exm',
                                       'first_name_first_name_lev_sim'])]




## Domanda

Partendo dalla regola data nella seguente funzione BlockingMatchinRule, dalla visualizzazione e valutazione dei relativi cluster, fare e commentare opportune modifiche alla funzione BlockingMatchinRule per migliorare precizione e recall. Si possono considerare  solo le FixedFeatures o eventualmente anche le altre disponibili

In [120]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['first_name'] + ' ' + A['last_name']
    B['mix'] = B['first_name'] + ' ' + B['last_name']

    C_SimJoin_mix1  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code'],
                                        r_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code']
                                     )
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['last_name_last_name_lev_sim(ltuple, rtuple) >= .7','zip_code_zip_code_exm(ltuple, rtuple) == 1'], FixedFeatures)
    brm.add_rule([ 'last_name_last_name_lev_sim(ltuple, rtuple) >= .3',
                  'first_name_first_name_lev_sim(ltuple, rtuple) >= .3' ], FixedFeatures)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

#    MT=C_SimJoin_mix

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [121]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

Numero Elementi 338
Cluster con max numero di elementi: [32]


,NumeroElementiPerCluster,NumeroCluster
0,1,162
1,2,71
2,3,10
3,4,1


In [122]:
# confronto con i cluster del gold standard
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)


Numero Elementi 338
Cluster con max numero di elementi: [4, 5, 6, 19, 22, 23, 27, 28]


,NumeroElementiPerCluster,NumeroCluster
0,1,158
1,2,78
2,3,8


In [123]:
# i cluster sono "mediamente più grandi" : nei calcolati ne abbiamo 10 da 3 e 1 da 4
# questo è indice della presenza di falsi negativi
# verifichiamolo con i match indotti

In [124]:
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati))

IndexError: positional indexers are out-of-bounds

In [ ]:
# vediamo qualche falso negativo
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_622_2403315,2_622_6223703,left_only,0_622_2403315,karen,palmer,female,57,ma,28314,2_622_6223703,karen,gollins,female,57,ma,28306
1,0_1140_6460300,2_1140_12513318,left_only,0_1140_6460300,amber,broach,female,32,sc,28115,2_1140_12513318,amber,smith,female,32,sc,27613
2,0_1240_1391987,1_1240_1340473,left_only,0_1240_1391987,teresa,starnes,female,63,nc,28570,1_1240_1340473,teresa,pender,female,63,nc,28570
3,1_1240_1340473,2_1240_13627651,left_only,1_1240_1340473,teresa,pender,female,63,nc,28570,2_1240_13627651,teresa,starnes,female,63,nc,27562
4,0_1740_13855498,2_1740_7342937,left_only,0_1740_13855498,kathren,woodham,female,38,,27530,2_1740_7342937,kathren,hayes,female,38,fl,28209
5,0_3140_907191,2_3140_1207977,left_only,0_3140_907191,angel,johnson,female,41,,28690,2_3140_1207977,angel,mcclellan,female,41,nc,28630
6,0_5640_563518,1_5640_3369336,left_only,0_5640_563518,amanda,conard,female,34,nc,28804,1_5640_3369336,amanda,waldrop,female,34,,27712
7,1_1340_3795261,2_1340_13807668,left_only,1_1340_3795261,denise,hines,female,37,,27801,2_1340_13807668,denise,wells,female,37,nc,27530
8,1_2622_9753093,2_2622_148884,left_only,1_2622_9753093,briana,lefler,female,26,nc,27231,2_2622_148884,briana,orr,female,26,nc,27253
9,1_2722_10836165,2_2722_8809482,left_only,1_2722_10836165,christine,jones,female,36,nc,28347,2_2722_8809482,christine,horton,female,36,nc,28373


In [ ]:
# vediamo qualche falso positivo
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,1_3822_11891631,2_2940_9137590,right_only,1_3822_11891631,david,troy,male,67,nc,28337,2_2940_9137590,david,brown,male,51,il,28409
1,0_3822_11913154,2_2940_9137590,right_only,0_3822_11913154,david,troy,male,73,nc,28337,2_2940_9137590,david,brown,male,51,il,28409
2,0_3551_2874784,1_1741_7343057,right_only,0_3551_2874784,christopher,hines,male,21,nc,27360,1_1741_7343057,christopher,miller,male,29,ga,28227
3,0_3751_1179043,1_5722_659298,right_only,0_3751_1179043,james,kelly,male,68,mi,28025,1_5722_659298,james,keefer,male,52,in,28748
4,0_1840_7447052,1_6141_7624860,right_only,0_1840_7447052,joseph,baker,male,46,,28270,1_6141_7624860,joseph,blumberg,male,38,nc,28269
5,1_6141_7624860,2_1840_11606714,right_only,1_6141_7624860,joseph,blumberg,male,38,nc,28269,2_1840_11606714,joseph,baker,male,46,va,28129
6,1_2722_10836165,2_4922_11476487,right_only,1_2722_10836165,christine,jones,female,36,nc,28347,2_4922_11476487,christian,banks,male,52,,28393
7,0_4922_6825581,1_2722_10836165,right_only,0_4922_6825581,christian,banks,male,52,oh,27524,1_2722_10836165,christine,jones,female,36,nc,28347
8,2_1122_68628,2_2240_934374,right_only,2_1122_68628,patricia,kurt,female,58,ny,27215,2_2240_934374,latricia,hunter,female,46,nc,28075
9,1_1122_9842647,2_2240_934374,right_only,1_1122_9842647,patricia,kurt,female,58,ny,27517,2_2240_934374,latricia,hunter,female,46,nc,28075


In [ ]:
# qui si individua il problema e si discutono possibili soluzioni

# dare più peso al last_name per i maschi ?



In [ ]:
# considerando  anche age nella rule cosa può migliorare ?

# Features da considerare nel matching
FixedFeatures = F[F.feature_name.isin(['last_name_last_name_lev_sim',
                                       'zip_code_zip_code_exm',
                                       'first_name_first_name_lev_sim', 'age_age_exm'])]


In [ ]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['first_name'] + ' ' + A['last_name']
    B['mix'] = B['first_name'] + ' ' + B['last_name']

    C_SimJoin_mix1  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code'],
                                        r_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code']
                                     )
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['last_name_last_name_lev_sim(ltuple, rtuple) >= .7','zip_code_zip_code_exm(ltuple, rtuple) == 1'], FixedFeatures)
    brm.add_rule([ 'last_name_last_name_lev_sim(ltuple, rtuple) >= .3',
                  'first_name_first_name_lev_sim(ltuple, rtuple) >= .3' ,'age_age_exm(ltuple, rtuple) == 1' ], FixedFeatures)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

#    MT=C_SimJoin_mix

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [ ]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 338
Cluster con max numero di elementi: [17, 19, 22, 23, 24, 26, 27]


,NumeroElementiPerCluster,NumeroCluster
0,1,175
1,2,71
2,3,7


In [ ]:
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,92,91,1,11,0.9891,0.8922,0.9381


In [ ]:
# vediamo i falsi negativi
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_622_2403315,2_622_6223703,left_only,0_622_2403315,karen,palmer,female,57,ma,28314,2_622_6223703,karen,gollins,female,57,ma,28306
1,0_1140_6460300,2_1140_12513318,left_only,0_1140_6460300,amber,broach,female,32,sc,28115,2_1140_12513318,amber,smith,female,32,sc,27613
2,0_1240_1391987,1_1240_1340473,left_only,0_1240_1391987,teresa,starnes,female,63,nc,28570,1_1240_1340473,teresa,pender,female,63,nc,28570
3,1_1240_1340473,2_1240_13627651,left_only,1_1240_1340473,teresa,pender,female,63,nc,28570,2_1240_13627651,teresa,starnes,female,63,nc,27562
4,0_1740_13855498,2_1740_7342937,left_only,0_1740_13855498,kathren,woodham,female,38,,27530,2_1740_7342937,kathren,hayes,female,38,fl,28209
5,0_3140_907191,2_3140_1207977,left_only,0_3140_907191,angel,johnson,female,41,,28690,2_3140_1207977,angel,mcclellan,female,41,nc,28630
6,0_5640_563518,1_5640_3369336,left_only,0_5640_563518,amanda,conard,female,34,nc,28804,1_5640_3369336,amanda,waldrop,female,34,,27712
7,1_1340_3795261,2_1340_13807668,left_only,1_1340_3795261,denise,hines,female,37,,27801,2_1340_13807668,denise,wells,female,37,nc,27530
8,1_2622_9753093,2_2622_148884,left_only,1_2622_9753093,briana,lefler,female,26,nc,27231,2_2622_148884,briana,orr,female,26,nc,27253
9,1_2722_10836165,2_2722_8809482,left_only,1_2722_10836165,christine,jones,female,36,nc,28347,2_2722_8809482,christine,horton,female,36,nc,28373


In [ ]:
# vediamo i falsi positivi
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_4651_6284216,2_622_6223703,right_only,0_4651_6284216,phillip,collins,male,54,nc,28306,2_622_6223703,karen,gollins,female,57,ma,28306


In [ ]:
# come fare una rule solo per le donne?

# dichiarare una features esplicitamente, seguendo il manuale ...

# ... oppure manualmente facendo un match solo per le donne

In [ ]:
FSOURCES = {}

for x in SOURCES.keys():
  X = deepcopy(SOURCES[x])
  FSOURCES[x] = X[X.sex=='female']


In [ ]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['first_name'] + ' ' + A['last_name'] + ' ' + A['zip_code'] + ' ' + A['birth_place']
    B['mix'] = B['first_name'] + ' ' + B['last_name'] + ' ' + B['zip_code'] + ' ' + B['birth_place']

    C_SimJoin_mix1  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.5,
                                        l_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code'],
                                        r_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code']
                                     )
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule([
                  'first_name_first_name_lev_sim(ltuple, rtuple) >= .95' ,'age_age_exm(ltuple, rtuple) == 1' ], FixedFeatures)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

#    MT=C_SimJoin_mix

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [ ]:
FMTSOURCES=MatchTableSOURCES(FSOURCES)
MTSOURCES=MatchTableSOURCES(SOURCES)
MTSOURCES=pd.concat([FMTSOURCES,MTSOURCES])

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


In [ ]:
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

Numero Elementi 338
Cluster con max numero di elementi: [2, 3, 5, 9, 31, 39]


,NumeroElementiPerCluster,NumeroCluster
0,1,186
1,2,67
2,3,6


In [ ]:
# vediamo i falsi negativi
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_622_2403315,2_622_6223703,left_only,0_622_2403315,karen,palmer,female,57,ma,28314,2_622_6223703,karen,gollins,female,57,ma,28306
1,0_1140_6460300,2_1140_12513318,left_only,0_1140_6460300,amber,broach,female,32,sc,28115,2_1140_12513318,amber,smith,female,32,sc,27613
2,1_1222_1104748,2_1222_962821,left_only,1_1222_1104748,leshondar,mcmullen,female,47,nc,28025,2_1222_962821,le shondra,mcmullen,female,47,,28025
3,0_1222_1130018,2_1222_962821,left_only,0_1222_1130018,leshondar,mcmullen,female,47,nc,28025,2_1222_962821,le shondra,mcmullen,female,47,,28025
4,0_1240_1391987,1_1240_1340473,left_only,0_1240_1391987,teresa,starnes,female,63,nc,28570,1_1240_1340473,teresa,pender,female,63,nc,28570
5,1_1240_1340473,2_1240_13627651,left_only,1_1240_1340473,teresa,pender,female,63,nc,28570,2_1240_13627651,teresa,starnes,female,63,nc,27562
6,0_1740_13855498,2_1740_7342937,left_only,0_1740_13855498,kathren,woodham,female,38,,27530,2_1740_7342937,kathren,hayes,female,38,fl,28209
7,0_2922_12142977,1_2922_8646097,left_only,0_2922_12142977,branden,lee,male,28,il,28079,1_2922_8646097,branden,lee,male,28,,28212
8,0_3140_907191,2_3140_1207977,left_only,0_3140_907191,angel,johnson,female,41,,28690,2_3140_1207977,angel,mcclellan,female,41,nc,28630
9,0_3822_11913154,1_3822_11891631,left_only,0_3822_11913154,david,troy,male,73,nc,28337,1_3822_11891631,david,troy,male,67,nc,28337


# Con dataset piccoli

Riconsideriamo lo stesso esempio precedente, ma con dataset più piccoli e meno attributi

In [ ]:
# DATASET CLEAN
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/PPRL/'

src_links = [
path+'cleanDatasetA3.csv',
path+'cleanDatasetB3.csv',
path+'cleanDatasetC3.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandard=pd.read_csv(path+ 'cleanGoldStandard3.csv')
GoldStandard.columns=['l_id','r_id']

In [ ]:
SOURCES['S3']

,id,given_name,surname,date_of_birth
0,S3_0,emmerson,loyck,19211129
1,S3_1,michel,wuchatsch,19190110
2,S3_2,joshua,green,19010219
3,S3_5,braedon,schuetz,19440909
4,S3_6,olivia,hobson,19760812
5,S3_7,joshua,green,19790110
6,S3_8,keely,clarke,19050410
7,S3_9,joshua,morriosn,19101123
8,S3_10,genoveffa,hylander,19071008
9,S3_11,genovefa,hyllande,19071008


In [ ]:
ClusterGoldStandardCLEAN=ClusterComponentiConnessi(GoldStandard[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
ClusterGoldStandardCLEAN.sample()

,ClusterKey,ClusterElement
5,3,S3_1


In [ ]:
def VisualizzaCluster(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[1]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS


In [ ]:
VisualizzaCluster(ClusterGoldStandardCLEAN)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,2,"1,3",2,"S1_0,S3_9"
1,2,2,"3,1",2,"S3_0,S1_2"
2,3,2,"1,3",2,"S1_4,S3_1"
3,4,3,"3,1,2",3,"S3_5,S1_6,S2_0"
4,5,2,"3,1",2,"S3_7,S1_7"
5,6,3,"3,1,2",3,"S3_6,S1_8,S2_4"
6,7,2,"2,1",2,"S2_7,S1_10"
7,8,2,"3,1",2,"S3_10,S1_11"
8,9,2,"2,3",2,"S2_5,S3_2"
9,10,1,3,1,S3_11


In [ ]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)

Numero Elementi 26
Cluster con max numero di elementi: [4, 6]


,NumeroElementiPerCluster,NumeroCluster
0,1,6
1,2,7
2,3,2


In [ ]:
# Consideriamo la loro unione nel dataframe UNIONE
# il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
# quindi effettuo unione tramite append
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
FeaturesList

['given_name_given_name_lev_dist',
 'given_name_given_name_lev_sim',
 'given_name_given_name_jar',
 'given_name_given_name_jwn',
 'given_name_given_name_exm',
 'given_name_given_name_jac_qgm_3_qgm_3',
 'surname_surname_jac_qgm_3_qgm_3',
 'surname_surname_cos_dlm_dc0_dlm_dc0',
 'surname_surname_jac_dlm_dc0_dlm_dc0',
 'surname_surname_mel',
 'surname_surname_lev_dist',
 'surname_surname_lev_sim',
 'surname_surname_nmw',
 'surname_surname_sw',
 'date_of_birth_date_of_birth_lev_dist',
 'date_of_birth_date_of_birth_lev_sim',
 'date_of_birth_date_of_birth_jar',
 'date_of_birth_date_of_birth_jwn',
 'date_of_birth_date_of_birth_exm',
 'date_of_birth_date_of_birth_jac_qgm_3_qgm_3']

In [ ]:
FixedFeatures=F

In [ ]:

def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['given_name'] + ' ' + A['surname']
    B['mix'] = B['given_name'] + ' ' + B['surname']

    C_SimJoin_mix1  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                        r_out_attrs=['given_name', 'surname', 'date_of_birth']
                                     )
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule([	'surname_surname_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.3'], FixedFeatures)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    #MT=C_SimJoin_mix1

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [ ]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)


0% [#####] 100% | ETA: 00:00:00

Numero Elementi 16
Cluster con max numero di elementi: [1, 2, 3, 4]



Total time elapsed: 00:00:00


,NumeroElementiPerCluster,NumeroCluster
0,1,8
1,2,4


In [ ]:
ClusterCalcolati.to_csv("ClusterCalcolati.csv", index=False)

In [ ]:
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati))


,MT,TP,FP,FN,P,R,F
0,16,12,4,1,0.75,0.9231,0.8276


In [ ]:
# vediamo qualche falso negativo
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id').head()

,l_id,r_id,_merge,id_x,given_name_x,surname_x,date_of_birth_x,id_y,given_name_y,surname_y,date_of_birth_y
0,S1_7,S3_7,left_only,S1_7,joshua,greenj,19790110,S3_7,joshua,green,19790110


In [ ]:
# vediamo qualche falso positivo
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id').head()

,l_id,r_id,_merge,id_x,given_name_x,surname_x,date_of_birth_x,id_y,given_name_y,surname_y,date_of_birth_y
0,S1_7,S2_5,right_only,S1_7,joshua,greenj,19790110,S2_5,joshua,green,19010219
1,S1_7,S3_2,right_only,S1_7,joshua,greenj,19790110,S3_2,joshua,green,19010219
2,S1_12,S3_11,right_only,S1_12,genoveffa,hyllander,19081008,S3_11,genovefa,hyllande,19071008
3,S1_5,S3_8,right_only,S1_5,amy,clarke,19940205,S3_8,keely,clarke,19050410


In [ ]:
## Nella parte di Blocking oltre al similarity join usare anche altre teniche di Blocking
# ad esempio Blocking Key sulle iniziali di nome e cognome

# dal notebook in cui è stato introdotto il blocking, si definisce prima tra due source

### Si calcolo la Blocking Key BK

dfA= SOURCES['S1']
dfB= SOURCES['S1']

# si settano le key dei due dataset
em.set_key(dfA, 'id')
em.set_key(dfB, 'id')

dfA['BK']=dfA.apply(lambda row: row['given_name'][:3] + row['surname'][:3], axis=1)
dfB['BK']=dfB.apply(lambda row: row['given_name'][:3] + row['surname'][:3], axis=1)


KeyError: ignored

In [ ]:
dfA

,id,given_name,surname,date_of_birth,BK
0,S1_0,joshua,morrison,19101123,josmor
1,S1_1,jordan,white,19371126,jorwhi
2,S1_2,emmerson,lock,19211129,emmloc
3,S1_4,michael,wuchatsch,19190110,micwuc
4,S1_5,amy,clarke,19940205,amycla
5,S1_6,rhys,schuetz,19440909,rhysch
6,S1_7,joshua,greenj,19790110,josgre
7,S1_8,olivia,hobson,19760812,olihob
8,S1_10,elisabett,domiten,19081008,elidom
9,S1_11,genoveffa,hylander,19071008,genhyl


In [ ]:

# Quindi si usa un Equivalence Blocker su BK

ab = em.AttrEquivalenceBlocker()
C = ab.block_tables(  dfA, dfB, # dataset
      'BK', 'BK', # si controlla A.BK==B.BK e si bloccano quelle che non soddisfano
      l_output_attrs=['BK','given_name', 'surname', 'date_of_birth'],
      r_output_attrs=['BK','given_name', 'surname', 'date_of_birth']
    )
C.head()

,_id,ltable_id,rtable_id,ltable_BK,ltable_given_name,ltable_surname,ltable_date_of_birth,rtable_BK,rtable_given_name,rtable_surname,rtable_date_of_birth
0,0,S1_0,S1_0,josmor,joshua,morrison,19101123,josmor,joshua,morrison,19101123
1,1,S1_1,S1_1,jorwhi,jordan,white,19371126,jorwhi,jordan,white,19371126
2,2,S1_2,S1_2,emmloc,emmerson,lock,19211129,emmloc,emmerson,lock,19211129
3,3,S1_4,S1_4,micwuc,michael,wuchatsch,19190110,micwuc,michael,wuchatsch,19190110
4,4,S1_5,S1_5,amycla,amy,clarke,19940205,amycla,amy,clarke,19940205


In [ ]:
C=C.rename(columns={'ltable_id': 'l_id', 'rtable_id': 'r_id'})
C

,_id,l_id,r_id,ltable_BK,ltable_given_name,ltable_surname,ltable_date_of_birth,rtable_BK,rtable_given_name,rtable_surname,rtable_date_of_birth
0,0,S1_0,S1_0,josmor,joshua,morrison,19101123,josmor,joshua,morrison,19101123
1,1,S1_1,S1_1,jorwhi,jordan,white,19371126,jorwhi,jordan,white,19371126
2,2,S1_2,S1_2,emmloc,emmerson,lock,19211129,emmloc,emmerson,lock,19211129
3,3,S1_4,S1_4,micwuc,michael,wuchatsch,19190110,micwuc,michael,wuchatsch,19190110
4,4,S1_5,S1_5,amycla,amy,clarke,19940205,amycla,amy,clarke,19940205
5,5,S1_6,S1_6,rhysch,rhys,schuetz,19440909,rhysch,rhys,schuetz,19440909
6,6,S1_7,S1_7,josgre,joshua,greenj,19790110,josgre,joshua,greenj,19790110
7,7,S1_8,S1_8,olihob,olivia,hobson,19760812,olihob,olivia,hobson,19760812
8,8,S1_10,S1_10,elidom,elisabett,domiten,19081008,elidom,elisabett,domiten,19081008
9,9,S1_11,S1_11,genhyl,genoveffa,hylander,19071008,genhyl,genoveffa,hylander,19071008


In [ ]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING

    A['BK']=A.apply(lambda row: row['given_name'][:3] + row['surname'][:3], axis=1)
    B['BK']=B.apply(lambda row: row['given_name'][:3] + row['surname'][:3], axis=1)

    ab = em.AttrEquivalenceBlocker()
    C = ab.block_tables(  A, B, # dataset
      'BK', 'BK', # si controlla A.BK==B.BK e si bloccano quelle che non soddisfano
      l_output_attrs=['BK','given_name', 'surname', 'date_of_birth'],
      r_output_attrs=['BK','given_name', 'surname', 'date_of_birth']
    )
    C=C.rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})

    print(C.columns)
    em.set_key(C, '_id')
    em.set_ltable(C, A)
    em.set_rtable(C, B)
    em.set_fk_ltable(C, 'l_id')
    em.set_fk_rtable(C, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule([	'surname_surname_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.3'], FixedFeatures)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]


    return MT

BlockingMatchingRule(SOURCES['S1'], SOURCES['S2'])

Index(['_id', 'l_id', 'r_id', 'ltable_BK', 'ltable_given_name',
       'ltable_surname', 'ltable_date_of_birth', 'rtable_BK',
       'rtable_given_name', 'rtable_surname', 'rtable_date_of_birth'],
      dtype='object')


,_id,l_id,r_id,ltable_BK,ltable_given_name,ltable_surname,ltable_date_of_birth,rtable_BK,rtable_given_name,rtable_surname,rtable_date_of_birth,pred_label
0,0,S1_7,S2_5,josgre,joshua,greenj,19790110,josgre,joshua,green,19010219,1
1,1,S1_8,S2_4,olihob,olivia,hobson,19760812,olihob,olivia,hobson,19760812,1
2,2,S1_10,S2_7,elidom,elisabett,domiten,19081008,elidom,elisabet,domitienn,19071008,1


# Esercizio (dirty)

Confrontare quanto fatto in precedenza nel caso clean con quello che si ottiene nel caso dirty: ripetere le stesse funzioni e fare opportune considerazioni

In [ ]:
# DATASET DIRTY
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/PPRL/'

src_links = [
path+'DATASET_small_dirty_A.csv',
path+'DATASET_small_dirty_B.csv',
path+'DATASET_small_dirty_C.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

In [ ]:
SOURCES['S1'].columns

Index(['rec_id', 'given_name', 'surname', 'date_of_birth', 'id'], dtype='object')

In [ ]:
# costruiamo il Gold Standard

# rinominiamo id in entity
for x in SOURCES.keys():
          SOURCES[x].columns=['id', 'given_name', 'surname', 'date_of_birth', 'entity']

# mettiamo tutto assieme
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])

# Calcoliamo il gold standard
GoldStandardDIRTY=UNIONE.merge(UNIONE, on='entity')[['id_x','id_y']].query('id_x <= id_y')

GoldStandardDIRTY.columns=['l_id','r_id']



In [ ]:
ClusterGoldStandardDIRTY=ClusterComponentiConnessi(GoldStandardDIRTY[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
ClusterGoldStandardDIRTY.sample()

,ClusterKey,ClusterElement
14,5,c_rec-1694-dup-1


In [ ]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandardDIRTY)


Numero Elementi 47
Cluster con max numero di elementi: [3, 5]


,NumeroElementiPerCluster,NumeroCluster
0,1,8
1,2,2
2,3,7
3,4,1
4,5,2


In [ ]:
def VisualizzaCluster(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS

VisualizzaCluster(ClusterGoldStandardDIRTY).sort_values("#Record", ascending=False)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
2,3,2,"a,c,a,a,c",5,"a_rec-1668-dup-2,c_rec-1668-dup-0,a_rec-1668-org,a_rec-1668-dup-3,c_rec-1668-dup-1"
4,5,2,"c,c,a,a,c",5,"c_rec-1694-dup-2,c_rec-1694-org,a_rec-1694-dup-0,a_rec-1694-dup-3,c_rec-1694-dup-1"
13,14,1,"b,b,b,b",4,"b_rec-1925-dup-2,b_rec-1925-dup-0,b_rec-1925-org,b_rec-1925-dup-3"
0,1,2,"a,c,a",3,"a_rec-1799-dup-0,c_rec-1799-dup-1,a_rec-1799-org"
16,17,2,"c,b,c",3,"c_rec-1560-dup-0,b_rec-1560-org,c_rec-1560-dup-4"
15,16,2,"c,b,c",3,"c_rec-1758-dup-1,b_rec-1758-org,c_rec-1758-dup-0"
6,7,3,"a,c,b",3,"a_rec-834-dup-1,c_rec-834-org,b_rec-834-dup-0"
7,8,2,"c,a,a",3,"c_rec-456-dup-1,a_rec-456-dup-0,a_rec-456-org"
8,9,3,"c,b,a",3,"c_rec-1657-org,b_rec-1657-dup-0,a_rec-1657-dup-4"
9,10,3,"a,b,c",3,"a_rec-1797-dup-1,b_rec-1797-org,c_rec-1797-dup-0"


In [ ]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['given_name'] + ' ' + A['surname']
    B['mix'] = B['given_name'] + ' ' + B['surname']

    C_SimJoin_mix1  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                        r_out_attrs=['given_name', 'surname', 'date_of_birth']
                                     )
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule([	'surname_surname_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.3'], FixedFeatures)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    #MT=C_SimJoin_mix1

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [ ]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

0% [############] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [#################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [#################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 47
Cluster con max numero di elementi: [1, 2, 3, 7]


,NumeroElementiPerCluster,NumeroCluster
0,1,13
1,2,11
2,3,4


In [ ]:
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandardDIRTY),CalcolaMatchIndottiCluster(ClusterCalcolati))



,MT,TP,FP,FN,P,R,F
0,23,15,8,34,0.6522,0.3061,0.4167


In [ ]:
# vediamo i falsi negativi
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardDIRTY),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,given_name_x,surname_x,date_of_birth_x,entity_x,id_y,given_name_y,surname_y,date_of_birth_y,entity_y
0,a_rec-1799-dup-0,c_rec-1799-dup-1,left_only,a_rec-1799-dup-0,joshua,morrison,19101123,rec-1799,c_rec-1799-dup-1,joshua,morriosn,19101123,rec-1799
1,a_rec-1799-dup-0,a_rec-1799-org,left_only,a_rec-1799-dup-0,joshua,morrison,19101123,rec-1799,a_rec-1799-org,joshua,morrison,19101123,rec-1799
2,a_rec-1668-dup-2,c_rec-1668-dup-0,left_only,a_rec-1668-dup-2,emmerson,lock,19211129,rec-1668,c_rec-1668-dup-0,lock,emmerson,19211129,rec-1668
3,a_rec-1668-org,c_rec-1668-dup-0,left_only,a_rec-1668-org,emmerson,lock,19211129,rec-1668,c_rec-1668-dup-0,lock,emmerson,19211129,rec-1668
4,a_rec-1668-dup-3,c_rec-1668-dup-0,left_only,a_rec-1668-dup-3,emmerson,lock,19211129,rec-1668,c_rec-1668-dup-0,lock,emmerson,19211129,rec-1668
5,a_rec-1668-dup-2,a_rec-1668-org,left_only,a_rec-1668-dup-2,emmerson,lock,19211129,rec-1668,a_rec-1668-org,emmerson,lock,19211129,rec-1668
6,a_rec-1668-dup-3,a_rec-1668-org,left_only,a_rec-1668-dup-3,emmerson,lock,19211129,rec-1668,a_rec-1668-org,emmerson,lock,19211129,rec-1668
7,a_rec-1668-dup-2,a_rec-1668-dup-3,left_only,a_rec-1668-dup-2,emmerson,lock,19211129,rec-1668,a_rec-1668-dup-3,emmerson,lock,19211129,rec-1668
8,a_rec-1668-dup-2,c_rec-1668-dup-1,left_only,a_rec-1668-dup-2,emmerson,lock,19211129,rec-1668,c_rec-1668-dup-1,emmerson,loyck,19211129,rec-1668
9,c_rec-1668-dup-0,c_rec-1668-dup-1,left_only,c_rec-1668-dup-0,lock,emmerson,19211129,rec-1668,c_rec-1668-dup-1,emmerson,loyck,19211129,rec-1668


In [ ]:
# vediamo i falsi positivi
VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandardDIRTY),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')


,l_id,r_id,_merge,id_x,given_name_x,surname_x,date_of_birth_x,entity_x,id_y,given_name_y,surname_y,date_of_birth_y,entity_y
0,b_rec-1654-org,c_rec-504-org,right_only,b_rec-1654-org,amy,clarke,19040510,rec-1654,c_rec-504-org,keely,clarke,19050410,rec-504
1,a_rec-1612-org,c_rec-504-org,right_only,a_rec-1612-org,amy,clarke,19940205,rec-1612,c_rec-504-org,keely,clarke,19050410,rec-504
2,a_rec-1612-org,b_rec-1654-org,right_only,a_rec-1612-org,amy,clarke,19940205,rec-1612,b_rec-1654-org,amy,clarke,19040510,rec-1654
3,a_rec-456-org,c_rec-1758-dup-1,right_only,a_rec-456-org,joshua,green,19790110,rec-456,c_rec-1758-dup-1,joshua,green,19010219,rec-1758
4,a_rec-456-org,b_rec-1758-org,right_only,a_rec-456-org,joshua,green,19790110,rec-456,b_rec-1758-org,joshua,green,19010219,rec-1758
5,a_rec-1225-org,b_rec-1925-dup-3,right_only,a_rec-1225-org,alexandra,grosser,19720305,rec-1225,b_rec-1925-dup-3,alexandra,grosvenor,19930305,rec-1925
6,a_rec-8160-org,c_rec-8106-org,right_only,a_rec-8160-org,genoveffa,hylander,19071008,rec-8160,c_rec-8106-org,genoveffa,hylander,19071008,rec-8106
7,a_rec-8170-org,c_rec-8134-org,right_only,a_rec-8170-org,genoveffa,hyllander,19081008,rec-8170,c_rec-8134-org,genovefa,hyllande,19071008,rec-8134


In [ ]:
# Consideriamo la loro unione nel dataframe UNIONE
# il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
# quindi effettuo unione tramite append
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])

In [ ]:
ClusterGoldStandardDIRTY=ClusterComponentiConnessi(GoldStandardDIRTY[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
ClusterGoldStandardDIRTY.sample()

,ClusterKey,ClusterElement
414,238,2_6231_8493344


In [ ]:
def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:1]
      x['ClusterElement']=x['ClusterElement'].astype(str)

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
ClusterGoldStandardDIRTY.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Elements', ascending=False)

,ClusterKey,#Sources,Sources,#Elements,Elements
0,1,2,"2,0",5,"2_22_3326652,2_22_91220,0_22_3235320,2_22_9930113,0_22_9865350"
1,2,2,"0,1",5,"0_40_13809878,0_40_9891855,1_40_13913995,0_40_12768214,0_40_13867121"
2,3,2,"0,1",4,"0_122_9112102,1_122_2397713,1_122_12738159,1_122_9000404"
3,4,3,"0,1,2",4,"0_140_13273677,0_140_9704280,1_140_9350108,2_140_12924542"
4,5,3,"0,1,2",4,"0_222_1749957,1_222_13226442,0_222_3198122,2_222_12748029"
...,...,...,...,...,...
154,155,1,1,1,1_4341_10364870
48,49,1,0,1,0_2651_8462590
144,145,1,1,1,1_3241_10003710
143,144,1,1,1,1_3141_13928423


In [ ]:
###### INSERIRE LA VOSTRA RISPOSTA A PARTIRE DA QUESTO PUNTO

# Esempio

E' in sostanza l'esempio precedente, con dati leggermente differenti.

Sintetizza il processo di Entity Resolution.






In [ ]:
# DATASET CLEAN
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/PPRL/'

src_links = [
path+'cleanDatasetA3.csv',
path+'cleanDatasetB3.csv',
path+'cleanDatasetC3.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandard=pd.read_csv(path+ 'cleanGoldStandard3.csv')

GoldStandard.columns=['l_id','r_id']

In [ ]:
ClusterGoldStandardCLEAN=ClusterComponentiConnessi(GoldStandard[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )

In [ ]:
def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:1]
      x['ClusterElement']=x['ClusterElement'].astype(str)

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
ClusterGoldStandardCLEAN.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Elements', ascending=False)

,ClusterKey,#Sources,Sources,#Elements,Elements
3,4,1,S,3,"S3_5,S2_0,S1_6"
5,6,1,S,3,"S3_6,S1_8,S2_4"
0,1,1,S,2,"S1_0,S3_9"
1,2,1,S,2,"S3_0,S1_2"
2,3,1,S,2,"S3_1,S1_4"
4,5,1,S,2,"S1_7,S3_7"
6,7,1,S,2,"S1_10,S2_7"
7,8,1,S,2,"S3_10,S1_11"
8,9,1,S,2,"S2_5,S3_2"
9,10,1,S,1,S3_11


In [ ]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)

Numero Elementi 26
Cluster con max numero di elementi: [4, 6]


,NumeroElementiPerCluster,NumeroCluster
0,1,6
1,2,7
2,3,2


In [ ]:
# si effettua l'unione di tutte le sources (ricordiamo che hanno tutte le stesso schema)
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
for x in SOURCES.keys():
        UNIONE=UNIONE.append(SOURCES[x])
# si generano le features tra Unione e se stessa
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })

em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
FeaturesList

['given_name_given_name_lev_dist',
 'given_name_given_name_lev_sim',
 'given_name_given_name_jar',
 'given_name_given_name_jwn',
 'given_name_given_name_exm',
 'given_name_given_name_jac_qgm_3_qgm_3',
 'surname_surname_jac_qgm_3_qgm_3',
 'surname_surname_cos_dlm_dc0_dlm_dc0',
 'surname_surname_jac_dlm_dc0_dlm_dc0',
 'surname_surname_mel',
 'surname_surname_lev_dist',
 'surname_surname_lev_sim',
 'surname_surname_nmw',
 'surname_surname_sw',
 'date_of_birth_date_of_birth_lev_dist',
 'date_of_birth_date_of_birth_lev_sim',
 'date_of_birth_date_of_birth_jar',
 'date_of_birth_date_of_birth_jwn',
 'date_of_birth_date_of_birth_exm',
 'date_of_birth_date_of_birth_jac_qgm_3_qgm_3']

In [ ]:
# Features da considerare nel matching
FixedFeatures = F[F.feature_name.isin(['surname_surname_jac_qgm_3_qgm_3',
                                       'given_name_given_name_jwn',
                                       'date_of_birth_date_of_birth_jac_qgm_3_qgm_3'])]

In [ ]:
# per verificare se vengono estratte correttamente
# si genera un candidate set tramite similarity join e si utilizza extract_feature_vecs
A['mix'] = A['given_name'] + ' ' + A['surname'] + ' ' + A['date_of_birth']
B['mix'] = B['given_name'] + ' ' + B['surname'] + ' ' + B['date_of_birth']


C_SimJoin_mix  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                        l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                        r_out_attrs=['given_name', 'surname', 'date_of_birth'])
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'l_l_id': 'l_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'r_r_id': 'r_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'_sim_score': 'sim'})

em.set_key(C_SimJoin_mix, '_id')
em.set_ltable(C_SimJoin_mix, A)
em.set_rtable(C_SimJoin_mix, B)
em.set_fk_ltable(C_SimJoin_mix, 'l_id')
em.set_fk_rtable(C_SimJoin_mix, 'r_id')


FeatureVector=em.extract_feature_vecs(C_SimJoin_mix,feature_table=FixedFeatures)
FeatureVector

0% [##########################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,_id,l_id,r_id,given_name_given_name_jwn,surname_surname_jac_qgm_3_qgm_3,date_of_birth_date_of_birth_jac_qgm_3_qgm_3
0,0,S1_0,S1_0,1.000000,1.000000,1.000000
1,1,S3_9,S1_0,1.000000,0.428571,1.000000
2,2,S1_1,S1_1,1.000000,1.000000,1.000000
3,3,S1_2,S1_2,1.000000,1.000000,1.000000
4,4,S3_0,S1_2,1.000000,0.444444,1.000000
...,...,...,...,...,...,...
65,65,S3_11,S3_10,0.977778,0.428571,1.000000
66,66,S1_11,S3_11,0.977778,0.428571,1.000000
67,67,S1_12,S3_11,0.977778,0.615385,0.538462
68,68,S3_10,S3_11,0.977778,0.428571,1.000000


In [ ]:
# in C_SimJoin_mix sim è quella di mix con sm.QgramTokenizer(qval=3)
C_SimJoin_mix

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim
0,0,S1_0,S1_0,joshua,morrison,19101123,joshua,morrison,19101123,1.000000
1,1,S3_9,S1_0,joshua,morriosn,19101123,joshua,morrison,19101123,0.733333
2,2,S1_1,S1_1,jordan,white,19371126,jordan,white,19371126,1.000000
3,3,S1_2,S1_2,emmerson,lock,19211129,emmerson,lock,19211129,1.000000
4,4,S3_0,S1_2,emmerson,loyck,19211129,emmerson,lock,19211129,0.814815
...,...,...,...,...,...,...,...,...,...,...
65,65,S3_11,S3_10,genovefa,hyllande,19071008,genoveffa,hylander,19071008,0.676471
66,66,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471
67,67,S1_12,S3_11,genoveffa,hyllander,19081008,genovefa,hyllande,19071008,0.611111
68,68,S3_10,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471


In [ ]:
pd.merge(C_SimJoin_mix, FeatureVector, on = ['l_id', 'r_id'])

,_id_x,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,_id_y,given_name_given_name_jwn,surname_surname_jac_qgm_3_qgm_3,date_of_birth_date_of_birth_jac_qgm_3_qgm_3
0,0,S1_0,S1_0,joshua,morrison,19101123,joshua,morrison,19101123,1.000000,0,1.000000,1.000000,1.000000
1,1,S3_9,S1_0,joshua,morriosn,19101123,joshua,morrison,19101123,0.733333,1,1.000000,0.428571,1.000000
2,2,S1_1,S1_1,jordan,white,19371126,jordan,white,19371126,1.000000,2,1.000000,1.000000,1.000000
3,3,S1_2,S1_2,emmerson,lock,19211129,emmerson,lock,19211129,1.000000,3,1.000000,1.000000,1.000000
4,4,S3_0,S1_2,emmerson,loyck,19211129,emmerson,lock,19211129,0.814815,4,1.000000,0.444444,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,65,S3_11,S3_10,genovefa,hyllande,19071008,genoveffa,hylander,19071008,0.676471,65,0.977778,0.428571,1.000000
66,66,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,66,0.977778,0.428571,1.000000
67,67,S1_12,S3_11,genoveffa,hyllander,19081008,genovefa,hyllande,19071008,0.611111,67,0.977778,0.615385,0.538462
68,68,S3_10,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,68,0.977778,0.428571,1.000000


# Esempio BIKE


In [ ]:
# DATASET CLEAN
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/BIKE/'

src_links = [
path+'BikeA.csv',
path+'BikeB.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandard=pd.read_csv(path+ 'BikeGS.csv')

GoldStandard.columns=['l_id','r_id']

In [ ]:
ClusterGoldStandard=ClusterComponentiConnessi(GoldStandard[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )

In [ ]:
def VisualizzaCluster(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS


In [ ]:
VisualizzaCluster(ClusterGoldStandard).head(3)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,2,"A,B",2,"A_37934,B_19529"
1,2,2,"B,A",2,"B_34527,A_40590"
2,3,2,"A,B",2,"A_41351,B_10617"


In [ ]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandard)

Numero Elementi 238
Cluster con max numero di elementi: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


,NumeroElementiPerCluster,NumeroCluster
0,1,218
1,2,10


In [ ]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# AttrEquivalenceBlocker:
    Attributi=['bike_name', 'city_posted', 'km_driven', 'price', 'color']
    ab = em.AttrEquivalenceBlocker()
    CandidateDato = ab.block_tables(
                          A, B, # dataset
            'color', 'color',
      l_output_attrs=Attributi,
      r_output_attrs=Attributi
    ).rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})
 #  print(len(CandidateDato))

 #   print(ValutaBlocking(A,B,CandidateDato[['l_id','r_id']],GoldStandard[['l_id','r_id']]))

    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
#   print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule([ 'bike_name_bike_name_jac_qgm_3_qgm_3(ltuple, rtuple) > .5' ], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
    print(len(predictions[predictions.pred_label==1]))
    MTDato=predictions[predictions.pred_label==1]

    return MTDato

In [ ]:
MT=BlockingMatchingRule(SOURCES['S1'],SOURCES['S2'])

print(Valuta(GoldStandard[['l_id','r_id']],MT[['l_id','r_id']]))

68
   MT  TP  FP  FN       P    R       F
0  68   2  66   8  0.0294  0.2  0.0513


In [ ]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# AttrEquivalenceBlocker:
    Attributi=['bike_name', 'city_posted', 'km_driven', 'price', 'color']
    ab = em.AttrEquivalenceBlocker()
    CandidateDato = ab.block_tables(
                          A, B, # dataset
            'km_driven', 'km_driven',
      l_output_attrs=Attributi,
      r_output_attrs=Attributi
    ).rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})
    print(len(CandidateDato))

    print(ValutaBlocking(A,B,CandidateDato[['l_id','r_id']],GoldStandard[['l_id','r_id']]))

    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
#    print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule([ 'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .5' ], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
    print(len(predictions[predictions.pred_label==1]))
    MTDato=predictions[predictions.pred_label==1]

    return MTDato

In [ ]:
MT=BlockingMatchingRule(SOURCES['S1'],SOURCES['S2'])


GoldStandard=MT[['l_id',	'r_id']]

print(Valuta(GoldStandard[['l_id','r_id']],MT[['l_id','r_id']]))



284
     A    B  BlockSize  ReductRatio  PCompletness  PQuality
0  117  121        284       0.9799           1.0    0.0352
10
   MT  TP  FP  FN    P    R    F
0  10  10   0   0  1.0  1.0  1.0


# Per Generare Esempi

Nell'esempio precedente il Gold Standard era stato creato esattamente con la funzione  BlockingMatchingRule(A,B) data, e quindi abbiamo ovviamente ottenuto
```
P    R    F
1.0  1.0  1.0
```



Quindi per provare altri esempi si può generare "casualmente" un Gold Standard e quindi fare altre strategie di entity resolution per valutarle rispetto a tale Gold Standard "casuale"

In [ ]:
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/BIKE/'

src_links = [
path+'BikeA.csv',
path+'BikeB.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

In [ ]:
def Genera(Regole):

    for regola  in Regole:
      print(regola)

    # Generiamo un nome da dare al Gold Standard per salvarlo poi successivamente
      posizioni = [Regole.index(rule) + 1 for rule in [regola]]
      stringa_posizioni = ''.join(map(str, posizioni))
      stringa_posizioni = stringa_posizioni+'.csv'
      print(f"Stringa xyz: {stringa_posizioni}\n")

      def BlockingMatchingRule(A, B):
        A = deepcopy(A)
        B = deepcopy(B)
        A = A.rename(columns={'id': "l_id"})
        B = B.rename(columns={'id': "r_id"})

        em.set_key(A, 'l_id')
        em.set_key(B, 'r_id')

    # BLOCKING by Similarity Join
        Attributi = ['bike_name', 'city_posted', 'km_driven', 'price', 'color']
        A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
        B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

        C_SimJoin_mix1  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=Attributi,
                                        r_out_attrs=Attributi
                                     )
        C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
        C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
        C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
        em.set_key(C_SimJoin_mix1, '_id')
        em.set_ltable(C_SimJoin_mix1, A)
        em.set_rtable(C_SimJoin_mix1, B)
        em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
        em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

        F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

        brm = em.BooleanRuleMatcher()
        brm.add_rule([regola], F)
        predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
        MTDato = predictions[predictions.pred_label == 1]

        return MTDato


      def MatchTableSOURCES(Sources: list):
        MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

        matching_option = random.choice(['stable_marriage', 'simmetric_best_match', 'none'])
        for x in Sources.keys():
            for y in Sources.keys():
                if (x < y):  # x<=y nel caso dirty

                    MTxy = BlockingMatchingRule(Sources[x], Sources[y])

                    print("Prima del global mapping ho questo numero di matching: ", len(MTxy))
                    if matching_option == 'stable_marriage':
                      MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
                      print(matching_option)
                    elif matching_option == 'simmetric_best_match':
                      MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))
                      print(matching_option)
                    print("Dopo l'eventuale  global mapping ho questo numero di matching: ", len(MTxy))

                    MatchTable = MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
        return MatchTable

      MT = MatchTableSOURCES(SOURCES)
      GoldStandard = MT[['l_id', 'r_id']]
      GoldStandard.to_csv(stringa_posizioni, index=False)

In [ ]:
RegoleDate = [
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .3',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .5',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .7',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + city_posted_city_posted_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.3',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + city_posted_city_posted_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.5',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + city_posted_city_posted_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.7',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.3',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.5',
      'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.7']

RegoleSelezionateCaso=random.sample(RegoleDate, 5)

Genera(RegoleSelezionateCaso)

bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.3
Stringa xyz: 1.csv



0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Prima del global mapping ho questo numero di matching:  1417
Dopo l'eventuale  global mapping ho questo numero di matching:  1417
bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .3
Stringa xyz: 2.csv



0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Prima del global mapping ho questo numero di matching:  602
Dopo l'eventuale  global mapping ho questo numero di matching:  602
bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .5
Stringa xyz: 3.csv



...

Prima del global mapping ho questo numero di matching:  164
stable_marriage
Dopo l'eventuale  global mapping ho questo numero di matching:  42
bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + city_posted_city_posted_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.5
Stringa xyz: 4.csv



0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Prima del global mapping ho questo numero di matching:  343
simmetric_best_match
Dopo l'eventuale  global mapping ho questo numero di matching:  49
bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.5
Stringa xyz: 5.csv



0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Prima del global mapping ho questo numero di matching:  291
stable_marriage
Dopo l'eventuale  global mapping ho questo numero di matching:  62


Se consideriamo l'ultimo caso
```
bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.5
Stringa xyz: 5.csv

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00

Prima del global mapping ho questo numero di matching:  291
stable_marriage
Dopo l'eventuale  global mapping ho questo numero di matching:  62
```



In [ ]:
# Considerando
GoldStandard = pd.read_csv("5.csv")
# Possiamo svolgere un classico esercizio come segue:
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/BIKE/'

src_links = [
path+'BikeA.csv',
path+'BikeB.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

ClusterGoldStandard=ClusterComponentiConnessi(GoldStandard[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterGoldStandard)

Numero Elementi 238
Cluster con max numero di elementi: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62]


,NumeroElementiPerCluster,NumeroCluster
0,1,114
1,2,62


In [ ]:
#Partendo dalla regola data nella seguente funzione BlockingMatchinRule, dalla visualizzazione e valutazione dei relativi cluster,
# fare e commentare opportune modifiche alla funzione BlockingMatchinRule per migliorare precizione e recall.
#Si possono considerare  solo le FixedFeatures o eventualmente anche le altre disponibili

def BlockingMatchingRule(A, B):
    A = deepcopy(A)
    B = deepcopy(B)
    A = A.rename(columns={'id': "l_id"})
    B = B.rename(columns={'id': "r_id"})

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

    # BLOCKING by Similarity Join
    Attributi = ['bike_name', 'city_posted', 'km_driven', 'price', 'color']
    A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

    C_SimJoin_mix1 = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                       'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                       l_out_attrs=Attributi,
                                       r_out_attrs=Attributi
                                    )
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
    ############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
    #    print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .7'], F)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MTDato = predictions[predictions.pred_label == 1]

    return MTDato

def MatchTableSOURCES(Sources: list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    matching_option = random.choice(['stable_marriage', 'simmetric_best_match', 'none'])
    for x in Sources.keys():
        for y in Sources.keys():
            if (x < y):  # x<=y nel caso dirty
                MTxy = BlockingMatchingRule(Sources[x], Sources[y])
                MatchTable = MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [ ]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 238
Cluster con max numero di elementi: [1]


,NumeroElementiPerCluster,NumeroCluster
0,1,193
1,2,3
2,3,1
3,4,4
4,5,1
5,6,1
6,9,1


In [ ]:
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,91,13,78,49,0.1429,0.2097,0.1699


In [ ]:
# ci sono molti FP nonostante una soglia molto elevata
#    brm.add_rule(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .7'], F)
# quindi applicchiamo mapping 1 1

def BlockingMatchingRule(A, B):
    A = deepcopy(A)
    B = deepcopy(B)
    A = A.rename(columns={'id': "l_id"})
    B = B.rename(columns={'id': "r_id"})

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

    # BLOCKING by Similarity Join
    Attributi = ['bike_name', 'city_posted', 'km_driven', 'price', 'color']
    A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

    C_SimJoin_mix1 = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                       'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                       l_out_attrs=Attributi,
                                       r_out_attrs=Attributi
                                    )
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
    ############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
    #    print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .7'], F)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MTDato = predictions[predictions.pred_label == 1]

    return MTDato


def MatchTableSOURCES(Sources: list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    matching_option = random.choice(['stable_marriage', 'simmetric_best_match', 'none'])
    for x in Sources.keys():
        for y in Sources.keys():
            if (x < y):  # x<=y nel caso dirty
                MTxy = BlockingMatchingRule(Sources[x], Sources[y])
                MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
                MatchTable = MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,MT,TP,FP,FN,P,R,F
0,15,13,2,49,0.8667,0.2097,0.3377


In [ ]:
### diminuiamo soglia
def BlockingMatchingRule(A, B):
    A = deepcopy(A)
    B = deepcopy(B)
    A = A.rename(columns={'id': "l_id"})
    B = B.rename(columns={'id': "r_id"})

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

    # BLOCKING by Similarity Join
    Attributi = ['bike_name', 'city_posted', 'km_driven', 'price', 'color']
    A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

    C_SimJoin_mix1 = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                       'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                       l_out_attrs=Attributi,
                                       r_out_attrs=Attributi
                                    )
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
    ############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
    #    print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .5'], F)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MTDato = predictions[predictions.pred_label == 1]

    return MTDato


def MatchTableSOURCES(Sources: list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    matching_option = random.choice(['stable_marriage', 'simmetric_best_match', 'none'])
    for x in Sources.keys():
        for y in Sources.keys():
            if (x < y):  # x<=y nel caso dirty
                MTxy = BlockingMatchingRule(Sources[x], Sources[y])
                MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
                MatchTable = MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,MT,TP,FP,FN,P,R,F
0,42,26,16,36,0.619,0.4194,0.5


In [ ]:
# per visualizzare i FN
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)

for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])

VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,bike_name_x,city_posted_x,km_driven_x,price_x,color_x,id_y,bike_name_y,city_posted_y,km_driven_y,price_y,color_y
0,A_40794,B_11383,left_only,A_40794,Bajaj Discover 100,Delhi,24000,28500,black,B_11383,Bajaj Discover 100 5 Speed,Delhi,23000,35000,black
1,A_46401,B_35856,left_only,A_46401,Royal Enfield Thunderbird 350,Bangalore,8900,13499,black,B_35856,Royal Enfield Thunderbird 350 Disc,Bangalore,9000,140000,grey
2,A_41836,B_22931,left_only,A_41836,TVS Star City Alloy,Delhi,26500,12999,black,B_22931,TVS Star City Alloy wheels,Delhi,26000,18000,grey
3,A_36734,B_15372,left_only,A_36734,Yamaha YZF R15,Delhi,17000,46000,red,B_15372,Yamaha YZF R15 V1,Delhi,30000,60000,black
4,A_42564,B_15249,left_only,A_42564,Yamaha SZ RR,Delhi,16000,34999,red,B_15249,Yamaha SZ Standard,Delhi,16900,40000,red
5,A_45737,B_8709,left_only,A_45737,Hero Honda Karizma R,Mumbai,19000,57999,black,B_8709,Hero Karizma R,Mumbai,18000,55000,white
6,A_43161,B_24925,left_only,A_43161,Yamaha SZ RR,Delhi,18000,44999,red,B_24925,Yamaha SZ R Standard,Delhi,8600,55000,red
7,A_40812,B_31966,left_only,A_40812,Hero Honda Karizma R,Delhi,38000,30000,red,B_31966,Hero Honda Hunk Standard,Delhi,44000,40000,red
8,A_37200,B_29224,left_only,A_37200,Royal Enfield Bullet Electra Twinspark,Bangalore,8700,110000,silver,B_29224,Royal Enfield Electra Standard,Delhi,21000,77000,silver
9,A_46543,B_22554,left_only,A_46543,Yamaha FZ S,Delhi,8700,49000,red,B_22554,Yamaha FZ S Standard,Delhi,19000,55000,black


In [ ]:
# vari falsi negativi hanno un colore diverso, quindi ha senso provare una regola che tenga tenga anche conto del colore, oppure anche considerare un blocking su color
def BlockingMatchingRule(A, B):
    A = deepcopy(A)
    B = deepcopy(B)
    A = A.rename(columns={'id': "l_id"})
    B = B.rename(columns={'id': "r_id"})

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

    # BLOCKING by Similarity Join
    Attributi = ['bike_name', 'city_posted', 'km_driven', 'price', 'color']
    A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

    C_SimJoin_mix1 = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                       'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                       l_out_attrs=Attributi,
                                       r_out_attrs=Attributi
                                    )
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
    ############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
    #    print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.6 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.4 > 0.5'], F)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MTDato = predictions[predictions.pred_label == 1]

    return MTDato


def MatchTableSOURCES(Sources: list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    matching_option = random.choice(['stable_marriage', 'simmetric_best_match', 'none'])
    for x in Sources.keys():
        for y in Sources.keys():
            if (x < y):  # x<=y nel caso dirty
                MTxy = BlockingMatchingRule(Sources[x], Sources[y])
                MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
                MatchTable = MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,MT,TP,FP,FN,P,R,F
0,85,50,35,12,0.5882,0.8065,0.6803


In [ ]:
# per visualizzare i FP
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)

for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])

VV=VediValuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id').sample(5)

,l_id,r_id,_merge,id_x,bike_name_x,city_posted_x,km_driven_x,price_x,color_x,id_y,bike_name_y,city_posted_y,km_driven_y,price_y,color_y
16,A_42564,B_24925,right_only,A_42564,Yamaha SZ RR,Delhi,16000,34999,red,B_24925,Yamaha SZ R Standard,Delhi,8600,55000,red
11,A_44547,B_17536,right_only,A_44547,Bajaj Pulsar 180cc Dtsi,Delhi,49000,29000,black,B_17536,Bajaj Pulsar 180 DTS- i Standard,Delhi,48000,24000,black
25,A_42793,B_15956,right_only,A_42793,Yamaha FZ16,Bangalore,10800,59999,black,B_15956,Yamaha Ray Standard,Bangalore,5500,54000,black
20,A_40886,B_11872,right_only,A_40886,Yamaha FZ FI Version 2.0,Bangalore,5700,79000,red,B_11872,Yamaha YZF R15 V 2.0,Bangalore,11000,110000,red
19,A_46415,B_24214,right_only,A_46415,Bajaj Pulsar 150cc Dtsi,Bangalore,38900,39999,black,B_24214,Bajaj Pulsar 150 DTS- i Standard,Bangalore,60000,40000,black


In [ ]:
# il colore è lo stesso ma il nome è abbastanza diverso,  aumentiamo peso del nome
def BlockingMatchingRule(A, B):
    A = deepcopy(A)
    B = deepcopy(B)
    A = A.rename(columns={'id': "l_id"})
    B = B.rename(columns={'id': "r_id"})

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

    # BLOCKING by Similarity Join
    Attributi = ['bike_name', 'city_posted', 'km_driven', 'price', 'color']
    A['mix'] = A[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)
    B['mix'] = B[Attributi].astype(str).apply(lambda row: ' '.join(row), axis=1)

    C_SimJoin_mix1 = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                       'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                       l_out_attrs=Attributi,
                                       r_out_attrs=Attributi
                                    )
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1 = C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    em.set_key(C_SimJoin_mix1, '_id')
    em.set_ltable(C_SimJoin_mix1, A)
    em.set_rtable(C_SimJoin_mix1, B)
    em.set_fk_ltable(C_SimJoin_mix1, 'l_id')
    em.set_fk_rtable(C_SimJoin_mix1, 'r_id')
    ############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
    #    print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple)*0.7 + color_color_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.5'], F)
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MTDato = predictions[predictions.pred_label == 1]

    return MTDato


def MatchTableSOURCES(Sources: list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    matching_option = random.choice(['stable_marriage', 'simmetric_best_match', 'none'])
    for x in Sources.keys():
        for y in Sources.keys():
            if (x < y):  # x<=y nel caso dirty
                MTxy = BlockingMatchingRule(Sources[x], Sources[y])
                MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
                MatchTable = MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,MT,TP,FP,FN,P,R,F
0,62,62,0,0,1.0,1.0,1.0
